# 第 70 课｜可信本地 PEFT Adapter 的最小训练闭环、发布与回滚

从第69课的“训练前必须审计”推进到“训练后如何安全保留、验证、上线与恢复”。

本课的主线不是夸大模型能力，而是建立**有身份、有版本、有验证、有开发门槛、可回滚的 adapter 工程流程**：

```text
检查基座和token边界
    ↓
只训练Q/V LoRA
    ↓
开发集选择候选（测试仍封存）
    ↓
staging保存仅含A/B权重
    ↓
全新模型独立重载、重新评分与文件哈希检查
    ↓
发布不可覆盖的版本目录 → 原子更新CURRENT
    ↓
回滚到基线 → 验证 → 恢复候选
    ↓
最后才查看留出测试
```

**两个独立可执行文件：** Notebook 含完整代码，不导入配套脚本；`.py` 可在命令行独立使用。默认CPU离线，不安装库、不下载模型、不运行远程自定义代码。

## 0. 环境与可验证边界

本课程默认路径使用 PyTorch 和标准库，模型是**随机初始化的教学 Transformer＋手写 LoRA**；它不是预训练Qwen，也**不把手写 toy adapter 假装成 PEFT 可直接加载的 adapter**。

但脚本另外提供**显式 `--local-peft-model` 路径**，用于已有、可信、符合检查条件的本地 Hugging Face Safetensors，调用真实 `transformers`、`peft`，只保存 `adapter_model.safetensors` 和 `adapter_config.json`，并独立重载验证。它要求这些依赖已由你自己安装，默认不会执行，在本次制作环境没有实际本地模型可供验证，因此不宣称实测过这个可选路径。

建议先从上到下运行所有默认单元；最后再独立完成三道练习。练习检查开关默认是 `False`。

失败发生在发布前，不能让旧 `CURRENT` 消失；失败发生在版本rename后、指针更新前，可能留下**未激活的版本目录**，但不应该错误激活。

本课只对**单写者、同一普通本地文件系统**实施尽力的临时文件、fsync和原子rename，不提供数据库ACID、多进程抢占锁或网络FS崩溃一致性保证；文件摘要和限制不构成针对恶意checkpoint的安全沙盒。

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""第70课：可信本地PEFT adapter最小闭环——验证、版本化发布与回滚。

默认无需联网或Transformers/PEFT，以随机微型因果Transformer+手写LoRA演练完整
训练->dev门控->adapter-only保存->独立重载->发布指针->回滚->重启恢复。
显式 --local-peft-model 才加载用户已有可信本地HF safetensors与PEFT；不自动下载、
不加载远程自定义代码，也不将教学随机模型成绩称为预训练模型成绩。
保存检验用于防误接/意外损坏，不构成对恶意仓库或并发网络文件系统的安全保证。
"""
from __future__ import annotations

import argparse
import copy
import hashlib
import importlib.util
import json
import math
import os
import platform
import random
import re
import struct
import sys
import tempfile
import time
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

try:
    import torch
    from torch import nn
    from torch.nn import functional as F
except ImportError as exc:
    raise SystemExit("需要已有PyTorch环境；本课不自动安装任何依赖。") from exc

VERSION = "lesson70-adapter-release-v1"
IGNORE = -100
SOURCES = {
    "peft_checkpoint": "https://huggingface.co/docs/peft/main/developer_guides/checkpoint",
    "peft_model": "https://huggingface.co/docs/peft/main/package_reference/peft_model",
    "peft_lora": "https://huggingface.co/docs/peft/main/package_reference/lora",
    "transformers_offline": "https://huggingface.co/docs/transformers/main/installation",
    "transformers_auto": "https://huggingface.co/docs/transformers/model_doc/auto",
    "chat_template": "https://huggingface.co/docs/transformers/main/chat_templating",
    "pytorch_checkpoint": "https://docs.pytorch.org/docs/stable/notes/serialization.html",
    "pytorch_repro": "https://docs.pytorch.org/docs/stable/notes/randomness.html",
}

class LessonError(ValueError):
    """拒绝含糊的token边界、停止状态、报告格式或资源预算。"""

def require(ok: bool, msg: str) -> None:
    if not bool(ok):
        raise LessonError(msg)

def canonical(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True,
                      separators=(",", ":"), allow_nan=False)

def digest(value: Any) -> str:
    return hashlib.sha256(canonical(value).encode("utf-8")).hexdigest()

def write_json(path: Path, value: Any) -> None:
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, sort_keys=True,
                               allow_nan=False) + "\n", encoding="utf-8")

def bounded_str(value: Any, length: int, label: str) -> str:
    require(type(value) is str and len(value.encode("utf-8")) <= length,
            label + "必须是有界字符串")
    return value

def strict_json(text: str):
    bounded_str(text, 8192, "JSON")
    def pairs(items):
        result = {}
        for key, value in items:
            require(key not in result, "重复JSON字段")
            result[key] = value
        return result
    def bad(_):
        raise LessonError("不接受NaN/Infinity")
    try:
        return json.loads(text, object_pairs_hook=pairs, parse_constant=bad)
    except (json.JSONDecodeError, RecursionError, UnicodeError) as exc:
        raise LessonError("JSON无法严格解析") from exc

def device_from_name(name: str):
    require(name in {"cpu", "cuda", "auto"}, "device应为cpu/cuda/auto")
    if name == "auto":
        name = "cuda" if torch.cuda.is_available() else "cpu"
    require(name != "cuda" or torch.cuda.is_available(), "CUDA不可用，不静默退回CPU")
    return torch.device(name)

def seed_all(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def state_hash(state: dict[str,torch.Tensor]) -> str:
    h=hashlib.sha256()
    for key,tensor in sorted(state.items()):
        t=tensor.detach().cpu().contiguous()
        h.update(canonical([key,str(t.dtype),list(t.shape)]).encode())
        h.update(t.reshape(-1).view(torch.uint8).numpy().tobytes())
    return h.hexdigest()

In [2]:
torch.set_num_threads(1)
DEVICE = 'cpu'
OUTPUT_ROOT = Path('lesson_70_outputs')
SEED = 7001
RANK, ALPHA = 4, 8.0
RUN_FULL_TRAINING = True
print('Python:',platform.python_version(),'torch:',torch.__version__)
print('CUDA可用:',torch.cuda.is_available(),'本课默认设备:',DEVICE)
print('课程版本:',VERSION)
print('教学seed:',SEED,'默认LoRA:',RANK,ALPHA)

Python: 3.13.5 torch: 2.10.0+cpu
CUDA可用: False 本课默认设备: cpu
课程版本: lesson70-adapter-release-v1
教学seed: 7001 默认LoRA: 4 8.0


## 1. 数据和模板：生成头与完整答案必须逐 token 对齐

任务仍为上一课建立的合成指令：

```text
system: Return only JSON with one string field named answer.
user:   copy:07,02
assistant: {"answer":"02"} + EOS
```

12个来源各4条；先按来源分成训练8、开发2、测试2，再形成 **32/8/8** 条。来源没有跨侧泄漏。模板由教学控制token显式建模，而不是把用户正文中的 `<assistant>` 字符串当成结构。

训练使用：

```
input_ids:  [BOS ... system ... user ... assistant_header ... answer ... EOS]
labels:     -100 ... -100 ... -100                  answer ... EOS
```

EOS是目标；prompt与padding不计入loss。模板前缀必须是完整token ID序列的**精确前缀**，不是从文本长度猜边界。真正HF模式使用本地tokenizer的 `apply_chat_template()`，不把这里的教学模板迁移到真实模型。

In [3]:
class ByteMergeChatTokenizer:
    """自建教学分词器：控制token独立于用户文本，普通文本按UTF-8字节合并。

    不假装实现HF的FastTokenizer或任意模型Chat Template。固定控制ID，避免把用户
    输入里的字面控制符误当结构token。真实应用必须使用对应模型的官方tokenizer/template。
    """
    pad_token_id = 0
    bos_token_id = 1
    eos_token_id = 2
    im_start_token_id = 3
    im_end_token_id = 4
    role_ids = {"system":5, "user":6, "assistant":7}
    byte_offset = 16
    merges = (b"OK", b" 42", b"42", b"hello", b"world", b"\n", b"answer", b"JSON", b"{\"answer\":")

    def __init__(self):
        self.merge_ids = {piece: self.byte_offset + 256 + i for i,piece in enumerate(self.merges)}
        self.ids_to_merge = {v:k for k,v in self.merge_ids.items()}
        self.vocab_size = self.byte_offset + 256 + len(self.merges)
        self.specials = {0:"<pad>",1:"<bos>",2:"<eos>",3:"<im_start>",
                         4:"<im_end>",5:"<system>",6:"<user>",7:"<assistant>"}

    def encode(self, text: str, *, add_special_tokens: bool=False) -> list[int]:
        bounded_str(text, 4096, "文本")
        raw = text.encode("utf-8")
        result = []
        index = 0
        while index < len(raw):
            matches = [piece for piece in self.merges if raw.startswith(piece, index)]
            if matches:
                piece = max(matches, key=len)
                result.append(self.merge_ids[piece]);index += len(piece)
            else:
                result.append(self.byte_offset + raw[index]);index += 1
        return ([self.bos_token_id] if add_special_tokens else []) + result

    def decode(self, ids: list[int], *, skip_special_tokens: bool=False) -> str:
        require(type(ids) is list, "token IDs必须是列表")
        raw = bytearray()
        for i in ids:
            require(type(i) is int and 0 <= i < self.vocab_size, "token ID越界")
            if i in self.specials:
                if skip_special_tokens:
                    continue
                raise LessonError("正文decode不允许控制token；传入skip_special_tokens=True")
            if i in self.ids_to_merge: raw.extend(self.ids_to_merge[i])
            elif self.byte_offset <= i < self.byte_offset + 256: raw.append(i - self.byte_offset)
            else: raise LessonError("保留ID不是合法正文token")
        try:
            return bytes(raw).decode("utf-8",errors="strict")
        except UnicodeDecodeError as exc:
            raise LessonError("不完整UTF-8回答；不要用replacement字符伪装成功") from exc

    def apply_chat_template(self, messages: list[dict], *, tokenize: bool=True,
                            add_generation_prompt: bool=False):
        require(type(messages) is list and bool(messages), "messages需要非空列表")
        require(type(add_generation_prompt) is bool, "generation prompt必须是bool")
        ids = [self.bos_token_id]
        for j,m in enumerate(messages):
            require(type(m) is dict and set(m)=={"role","content"}, "message字段必须恰好role/content")
            require(m["role"] in self.role_ids and type(m["content"]) is str, "角色或内容非法")
            require(not (j == len(messages)-1 and add_generation_prompt and m["role"]=="assistant"),
                    "若最后已有assistant，请使用续写语义，不要再添加generation prompt")
            ids.extend([self.im_start_token_id,self.role_ids[m["role"]]])
            ids.extend(self.encode(m["content"],add_special_tokens=False))
            ids.append(self.eos_token_id if m["role"]=="assistant" else self.im_end_token_id)
        if add_generation_prompt:
            ids.extend([self.im_start_token_id,self.role_ids["assistant"]])
        if tokenize:
            return ids
        # 仅用于人类阅读，控制符是标记，不要将其当普通文本用encode()重新编码。
        visible = []
        for i in ids:
            if i in self.specials: visible.append(self.specials[i])
            else: visible.append(self.decode([i],skip_special_tokens=True) if
                                 self._token_is_full_utf8(i) else "<byte-part>")
        return "".join(visible)

    def _token_is_full_utf8(self, i):
        if i in self.ids_to_merge:
            try: self.ids_to_merge[i].decode("utf-8");return True
            except UnicodeError: return False
        return self.byte_offset <= i < self.byte_offset + 128

    def contract(self) -> dict:
        return {"type":"toy-byte-merge", "specials":self.specials,
                "merges":[p.hex() for p in self.merges],"vocab":self.vocab_size,
                "prefix":"BOS+message start/role/body+end+assistant header",
                "eos":self.eos_token_id,"pad":self.pad_token_id}

SYSTEM_TEXT = 'Return only JSON with one string field named answer.'

def make_dataset(seed: int=6901) -> dict[str,list[dict]]:
    """12个来源×4条；来源分组8/2/2，不能拿测试来源调学习率。"""
    source_ids=list(range(12))
    random.Random(seed).shuffle(source_ids)
    part={a:('train' if i<8 else 'dev' if i<10 else 'test')
          for i,a in enumerate(source_ids)}
    groups={'train':[], 'dev':[], 'test':[]}
    for a in range(12):
        for b in range(4):
            row={'id':f's{a:02d}b{b:02d}', 'source_id':f's{a:02d}',
                 'split':part[a], 'system':SYSTEM_TEXT,
                 'user':f'copy:{a:02d},{b:02d}',
                 'assistant':canonical({'answer':f'{b:02d}'})}
            groups[part[a]].append(row)
    validate_dataset(groups)
    return groups

def validate_dataset(groups: dict[str,list[dict]]) -> None:
    require(type(groups) is dict and set(groups)=={'train','dev','test'}, '需要train/dev/test')
    seen_ids=set();seen_user=set();seen_sources={}
    for split,rows in groups.items():
        require(type(rows) is list and bool(rows),'划分不能为空')
        for row in rows:
            require(type(row) is dict and set(row)=={
                'id','source_id','split','system','user','assistant'},'记录字段不匹配')
            require(all(type(v) is str and v for v in row.values()),'所有字段必须为非空字符串')
            require(row['split']==split,'记录划分标签与所属集合不一致')
            require(row['id'] not in seen_ids and row['user'] not in seen_user,'ID/prompt重复')
            seen_ids.add(row['id']);seen_user.add(row['user'])
            require(row['system']==SYSTEM_TEXT,'系统指令不是固定版本')
            require(row['source_id']==row['id'][:3], '来源ID不一致')
            require(seen_sources.get(row['source_id'],split)==split,'相同来源跨划分')
            seen_sources[row['source_id']]=split
            prefix=row['user'].split(',')
            require(len(prefix)==2 and prefix[0]==f"copy:{int(row['source_id'][1:]):02d}",
                    '用户提示不符合教学固定格式')
            require(len(prefix[1])==2 and prefix[1].isascii() and prefix[1].isdigit(),
                    '复制源不是两位数字')
            require(row['assistant']==canonical({'answer':prefix[1]}),
                    '参考答案与复制任务规则不符')
    require([len(groups[s]) for s in ('train','dev','test')]==[32,8,8],
            '固定数据分组大小改变，不能直接比较原实验')

def messages_for_row(row: dict) -> list[dict]:
    return [{'role':'system','content':row['system']},
            {'role':'user','content':row['user']}]

def encode_training_row(tok: ByteMergeChatTokenizer,row: dict,
                        max_length: int=160) -> dict:
    require(type(max_length) is int and max_length>0,'长度上限非法')
    prompt=tok.apply_chat_template(messages_for_row(row),tokenize=True,
                                    add_generation_prompt=True)
    all_messages=messages_for_row(row)+[{'role':'assistant','content':row['assistant']}]
    full=tok.apply_chat_template(all_messages,tokenize=True,
                                add_generation_prompt=False)
    require(len(full)>len(prompt) and full[:len(prompt)]==prompt,
            'Chat Template的生成头/训练序列不具有精确token前缀关系')
    answer_ids=full[len(prompt):]
    require(answer_ids[-1]==tok.eos_token_id,'每条回答必须包含被监督的真实EOS')
    require(tok.eos_token_id not in answer_ids[:-1], '回答中间不允许结构EOS')
    require(all(i not in tok.specials for i in answer_ids[:-1]),'回答正文含结构token')
    require(len(full)<=max_length,'序列超过上限；不静默截断回答/EOS')
    require(len(prompt)>=2 and len(answer_ids)>=2,'提示或回答不完整')
    labels=[IGNORE]*len(prompt)+answer_ids
    return {'input_ids':full,'labels':labels,'prompt_len':len(prompt),
            'answer_len':len(answer_ids),'id':row['id']}

def collate_training_rows(encoded: list[dict],tok: ByteMergeChatTokenizer,
                          device='cpu') -> dict:
    require(type(encoded) is list and bool(encoded),'空批次')
    width=max(len(x['input_ids']) for x in encoded)
    ids=torch.full((len(encoded),width),tok.pad_token_id,dtype=torch.long,device=device)
    labels=torch.full_like(ids,IGNORE)
    mask=torch.zeros_like(ids,dtype=torch.bool)
    for n,row in enumerate(encoded):
        size=len(row['input_ids'])
        require(size>1 and size==len(row['labels']),'输入与标签不对齐')
        require(row['labels'][0]==IGNORE,'BOS不能被本课作为目标')
        require(row['labels'][-1]==tok.eos_token_id,'EOS未被监督')
        ids[n,:size]=torch.tensor(row['input_ids'],device=device)
        labels[n,:size]=torch.tensor(row['labels'],device=device)
        mask[n,:size]=True
    return {'input_ids':ids,'labels':labels,'attention_mask':mask}

In [4]:
tok=ByteMergeChatTokenizer()
data=make_dataset(SEED)
print('划分:',{s:len(rows) for s,rows in data.items()})
row=data['train'][0]
item=encode_training_row(tok,row)
print('示例来源/题目/正确答案:',row['source_id'],row['user'],row['assistant'])
print('prompt长度:',item['prompt_len'],'答案含EOS的长度:',item['answer_len'])
print('标签前缀全忽略:',all(x==IGNORE for x in item['labels'][:item['prompt_len']]))
print('最后目标是EOS:',item['labels'][-1]==tok.eos_token_id)

划分: {'train': 32, 'dev': 8, 'test': 8}
示例来源/题目/正确答案: s00 copy:00,00 {"answer":"00"}
prompt长度: 63 答案含EOS的长度: 7
标签前缀全忽略: True
最后目标是EOS: True


## 2. 真实的因果模型，但只让LoRA更新

默认：2层因果Transformer，隐藏维度32，4个注意力头，最大长度160。LoRA只注入每层Q/V的 `nn.Linear`，`rank=4, alpha=8`，A随机小值、B零初始化。

因此注入后初始输出应与冻结基座一致。更新只含A/B，embedding、LayerNorm和输出头不在optimizer中。训练前、发布前、重载后，均计算基座内容指纹。

“同结构、同rank”不能证明adapter来自正确基座。**同一个adapter叠加到不同权重，结果不是原模型的可靠复现。**

真实PEFT文档也明确说明 adapter-only 文件**依赖原始基座模型**，并不是可独立推理的完整模型。[PEFT checkpoint 格式](https://huggingface.co/docs/peft/main/developer_guides/checkpoint)

In [5]:
@dataclass(frozen=True)
class ToyConfig:
    vocab_size: int
    dim: int=32
    heads: int=4
    layers: int=2
    ff_dim: int=64
    max_length: int=160

    def validate(self):
        require(all(type(x) is int and x>0 for x in asdict(self).values()),'配置必须为正整数')
        require(self.dim%self.heads==0 and self.vocab_size>16,
                'dim/head或词表不匹配')

class CausalSelfAttention(nn.Module):
    def __init__(self,cfg: ToyConfig):
        super().__init__()
        d=cfg.dim
        self.heads=cfg.heads
        self.q_proj=nn.Linear(d,d,bias=False)
        self.k_proj=nn.Linear(d,d,bias=False)
        self.v_proj=nn.Linear(d,d,bias=False)
        self.o_proj=nn.Linear(d,d,bias=False)

    def forward(self,x:torch.Tensor,attention_mask:torch.Tensor):
        b,t,d=x.shape;h=self.heads
        def split(layer):
            return layer(x).reshape(b,t,h,d//h).transpose(1,2)
        q,k,v=split(self.q_proj),split(self.k_proj),split(self.v_proj)
        scores=(q@k.transpose(-1,-2))/math.sqrt(d//h)
        lower=torch.ones(t,t,device=x.device,dtype=torch.bool).tril()
        allowed=lower[None,None,:,:]&attention_mask[:,None,None,:]
        weights=torch.softmax(scores.masked_fill(~allowed,float('-inf')),dim=-1)
        result=(weights@v).transpose(1,2).contiguous().reshape(b,t,d)
        return self.o_proj(result)

class TransformerBlock(nn.Module):
    def __init__(self,cfg:ToyConfig):
        super().__init__()
        self.norm1=nn.LayerNorm(cfg.dim)
        self.attn=CausalSelfAttention(cfg)
        self.norm2=nn.LayerNorm(cfg.dim)
        self.ff=nn.Sequential(nn.Linear(cfg.dim,cfg.ff_dim),nn.GELU(),
                              nn.Linear(cfg.ff_dim,cfg.dim))

    def forward(self,x,mask):
        x=x+self.attn(self.norm1(x),mask)
        return x+self.ff(self.norm2(x))

class TinyChatLM(nn.Module):
    def __init__(self,cfg:ToyConfig):
        super().__init__();cfg.validate();self.cfg=cfg
        self.embed=nn.Embedding(cfg.vocab_size,cfg.dim)
        self.position=nn.Embedding(cfg.max_length,cfg.dim)
        self.blocks=nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.layers)])
        self.norm=nn.LayerNorm(cfg.dim)
        self.head=nn.Linear(cfg.dim,cfg.vocab_size,bias=False)

    def forward(self,input_ids:torch.Tensor,attention_mask:torch.Tensor):
        require(input_ids.ndim==2 and input_ids.dtype==torch.long,'输入必须为[B,T] long')
        b,t=input_ids.shape
        require(b>0 and t>1 and t<=self.cfg.max_length,'形状或上下文超限')
        require(attention_mask.shape==input_ids.shape and attention_mask.dtype==torch.bool,
                'attention mask形状/type不符')
        require(bool(attention_mask[:,0].all()),'不接受左补齐/空提示')
        require(not bool(((~attention_mask[:,:-1])&attention_mask[:,1:]).any()),
                '只允许右补齐')
        positions=torch.arange(t,device=input_ids.device)
        x=self.embed(input_ids)+self.position(positions)[None]
        for layer in self.blocks:x=layer(x,attention_mask)
        return self.head(self.norm(x))

class LoRALinear(nn.Module):
    def __init__(self,base:nn.Linear,rank=4,alpha=8.0):
        super().__init__()
        require(type(base) is nn.Linear and type(rank) is int and rank>0,
                '手写LoRA仅支持nn.Linear与正整数rank')
        require(type(alpha) in (int,float) and math.isfinite(alpha) and alpha>0,
                'LoRA alpha非法')
        self.base=base
        self.base.requires_grad_(False)
        self.rank=rank;self.alpha=float(alpha);self.enabled=True
        self.A=nn.Parameter(base.weight.new_empty(rank,base.in_features))
        self.B=nn.Parameter(base.weight.new_zeros(base.out_features,rank))
        nn.init.kaiming_uniform_(self.A,a=math.sqrt(5))

    def forward(self,x):
        result=self.base(x)
        if not self.enabled:return result
        return result+(self.alpha/self.rank)*F.linear(F.linear(x,self.A),self.B)

def inject_lora(base:TinyChatLM,rank=4,alpha=8.0) -> TinyChatLM:
    require(not any(isinstance(m,LoRALinear) for m in base.modules()),
            '不允许重复注入LoRA')
    policy=copy.deepcopy(base)
    policy.requires_grad_(False)
    for block in policy.blocks:
        block.attn.q_proj=LoRALinear(block.attn.q_proj,rank,alpha)
        block.attn.v_proj=LoRALinear(block.attn.v_proj,rank,alpha)
    return policy

def adapters(model:TinyChatLM):
    return [(n,p) for n,p in model.named_parameters() if n.endswith(('.A','.B'))]

def adapter_snapshot(model:TinyChatLM):
    return {n:p.detach().cpu().clone() for n,p in adapters(model)}

def restore_adapter(model:TinyChatLM,values:dict):
    parameters=dict(model.named_parameters())
    require(set(values)==set(adapter_snapshot(model)),'adapter键集合不匹配')
    with torch.no_grad():
        for name,v in values.items():
            require(v.shape==parameters[name].shape and bool(torch.isfinite(v).all()),
                    'adapter参数形状/数值不匹配')
            parameters[name].copy_(v.to(parameters[name]))

def base_state(model:TinyChatLM):
    base={}
    for name,value in model.state_dict().items():
        if name.endswith(('.A','.B')):continue
        normalized=name.replace('.q_proj.base.','.q_proj.').replace('.v_proj.base.','.v_proj.')
        require(normalized not in base,'归一化基座命名冲突')
        base[normalized]=value
    return base

def base_hash(model:TinyChatLM):
    return state_hash(base_state(model))

In [6]:
seed_all(SEED)
base_example=TinyChatLM(ToyConfig(tok.vocab_size)).requires_grad_(False).eval()
policy_example=inject_lora(base_example,RANK,ALPHA)
print('基座参数:',sum(p.numel() for p in base_example.parameters()))
print('可训练A/B:',sum(p.numel() for _,p in adapters(policy_example)))
print('trainable是否恰好A/B:',all(p.requires_grad==n.endswith(('.A','.B')) for n,p in policy_example.named_parameters()))

基座参数: 40000
可训练A/B: 1024
trainable是否恰好A/B: True


## 3. 训练损失与独立评估：不能只比较最后一个batch

模型只返回 `[B,T,V]` logits，所以训练函数**在外部移位一次**：`logits[:,:-1]` 对齐 `labels[:,1:]`。只对有效答案token求和，再按实际答案token数量取平均。不能先每题平均再对题目平均，除非你明确要换目标。

候选按开发集 NLL 选择，包括第0步；默认训练32步，每8步评估一次开发集。只保存“开发集NLL最低”的候选快照，不根据测试结果挑轮。训练结果取决于学习率、模型初始化和合成任务；本课默认数值是用于演示工作流的固定设置，不是通用建议。

此外，teacher-forced NLL与自由生成正确率分别记录：给定真实前缀的评分不等于贪心生成一定正确。

In [7]:
def supervised_token_loss(logits:torch.Tensor,labels:torch.Tensor) -> tuple[torch.Tensor,dict]:
    """模型仅返回logits，在此处shift一次，返回token平均SFT loss。"""
    require(logits.ndim==3 and labels.shape==logits.shape[:2] and labels.dtype==torch.long,
            'logits/labels形状type错误')
    require(logits.shape[1]>1 and bool((labels[:,0]==IGNORE).all()),'首位置必须忽略')
    require(bool(torch.isfinite(logits).all()),'非有限logits')
    targets=labels[:,1:];valid=targets!=IGNORE
    count=int(valid.sum())
    require(count>0,'全忽略不能训练')
    require(bool(((targets[valid]>=0)&(targets[valid]<logits.shape[-1])).all()),
            '监督目标越界')
    # 显式记录分子/分母，支持变长batch和累积训练时正确归一化。
    token_losses=F.cross_entropy(logits[:,:-1,:].reshape(-1,logits.shape[-1]),
                                 targets.reshape(-1),ignore_index=IGNORE,reduction='none')
    total=token_losses.sum();loss=total/count
    return loss,{'target_tokens':count,'loss_sum':total.detach().item(),
                 'mean_nll':loss.detach().item()}

def memory_accounting(policy:TinyChatLM,batch_size:int,seq_length:int)->dict:
    """仅为静态分项核算，不是训练峰值的上下界承诺。"""
    require(type(batch_size)is int and batch_size>0 and type(seq_length)is int and seq_length>0,
            'batch/seq必须正整数')
    base_bytes=sum(p.numel()*p.element_size() for p in policy.parameters() if not p.requires_grad)
    adapter_bytes=sum(p.numel()*p.element_size() for _,p in adapters(policy))
    grad_bytes=adapter_bytes
    # FP32 Adam状态的两个一阶/二阶矩仅是示例，另有可能的临时工作区。
    adam_moments=2*adapter_bytes
    b,t=batch_size,seq_length
    activations_proxy=b*t*policy.cfg.dim*4
    attention_proxy=b*policy.cfg.heads*t*t*4*policy.cfg.layers
    output_proxy=b*t*policy.cfg.vocab_size*4
    static=base_bytes+adapter_bytes+grad_bytes+adam_moments
    return {'dtype':'fp32_toy','frozen_parameter_bytes':base_bytes,
            'trainable_parameter_bytes':adapter_bytes,'adapter_gradient_bytes':grad_bytes,
            'estimated_adam_moment_bytes':adam_moments,
            'bookkept_static_bytes':static,'bookkept_static_mib':static/2**20,
            'activation_example_bytes':activations_proxy,
            'attention_example_bytes':attention_proxy,
            'logits_example_bytes':output_proxy,
            'warning':'static accounting/illustrative activation sizes; NOT measured peak, upper bound or GPU reservation'}

@torch.no_grad()
def evaluate_nll(model:TinyChatLM,rows:list[dict],tok:ByteMergeChatTokenizer,
                 max_length=160,batch_size=8)->dict:
    require(bool(rows),'评估集合不能空')
    mode=model.training;model.eval()
    device=next(model.parameters()).device
    total_loss=0.0;total_tokens=0
    try:
        for start in range(0,len(rows),batch_size):
            chunk=[encode_training_row(tok,r,max_length) for r in rows[start:start+batch_size]]
            b=collate_training_rows(chunk,tok,device)
            value,parts=supervised_token_loss(model(b['input_ids'],b['attention_mask']),b['labels'])
            total_loss+=parts['loss_sum'];total_tokens+=parts['target_tokens']
    finally:model.train(mode)
    return {'examples':len(rows),'target_tokens':total_tokens,
            'teacher_forced_nll':total_loss/total_tokens,
            'not_generation_accuracy':True}

def toy_adapter_step(policy:TinyChatLM,optimizer,rows:list[dict],
                     tok:ByteMergeChatTokenizer,max_length=160,max_grad_norm=1.0) -> dict:
    require(all(r['split']=='train' for r in rows),'更新入口只接受train')
    trainable={id(p) for _,p in adapters(policy)}
    optimizer_params=[p for group in optimizer.param_groups for p in group['params']]
    require(len(optimizer_params)==len(trainable) and
            {id(p) for p in optimizer_params}==trainable,
            '优化器必须恰好含LoRA A/B，不可含冻结基座')
    require(all(p.requires_grad == n.endswith(('.A','.B'))
                for n,p in policy.named_parameters()),'只有LoRA A/B允许梯度')
    require(type(max_grad_norm) in (int,float) and math.isfinite(max_grad_norm)
            and max_grad_norm>0,'梯度阈值非法')
    device=next(policy.parameters()).device
    encoded=[encode_training_row(tok,r,max_length) for r in rows]
    batch=collate_training_rows(encoded,tok,device)
    policy.train()
    optimizer.zero_grad(set_to_none=True)
    loss,parts=supervised_token_loss(policy(batch['input_ids'],batch['attention_mask']),batch['labels'])
    loss.backward()
    gradients={n:float(p.grad.norm()) for n,p in adapters(policy) if p.grad is not None}
    require(len(gradients)==len(trainable) and all(math.isfinite(x) for x in gradients.values()),
            'LoRA梯度未连通或包含非有限值')
    require(all(p.grad is None for n,p in policy.named_parameters() if not n.endswith(('.A','.B'))),
            '冻结基座收到梯度')
    norm=nn.utils.clip_grad_norm_([p for _,p in adapters(policy)],
                                   max_grad_norm,error_if_nonfinite=True)
    optimizer.step()
    return {'loss':float(loss.detach()),'tokens':parts['target_tokens'],
            'grad_norm_pre_clip':float(norm),'A_grad_sum':sum(v for k,v in gradients.items() if k.endswith('.A')),
            'B_grad_sum':sum(v for k,v in gradients.items() if k.endswith('.B'))}

In [8]:
seed_all(7099)
small_base=TinyChatLM(ToyConfig(tok.vocab_size,dim=16,heads=2,layers=1,ff_dim=32)).requires_grad_(False).eval()
small_policy=inject_lora(small_base,2,4.)
print('toy开发集初始NLL:',evaluate_nll(small_policy,data['dev'],tok)['teacher_forced_nll'])
print('生成函数稍后在第5节定义；这里不提前调用未定义函数。')
print('静态内存记账不是训练峰值上界：',memory_accounting(small_policy,4,160)['bookkept_static_mib'],'MiB')

toy开发集初始NLL: 6.227367401123047
生成函数稍后在第5节定义；这里不提前调用未定义函数。
静态内存记账不是训练峰值上界： 0.05438232421875 MiB


## 4. 真实本地 Safetensors 的预检（默认不加载）

这一段沿用前课的受信任文件检查，包括配置、权重分片索引、全文件 SHA-256、是否是 decoder-only、tokenizer文件证据、符号链接、字节上限以及输出目录是否在模型目录外。

这是**先检查再允许加载**的资源闸门，不是对任意恶意文件的保证。即使允许 `trust_remote_code=False`，加载用户提供的模型与依赖仍应限于可信环境。[Transformers local_files_only / trust_remote_code](https://huggingface.co/docs/transformers/model_doc/auto)

本课默认不会调用任何 `AutoModelForCausalLM.from_pretrained()`；离线Toy训练不会使用这些外部文件。

In [9]:
def stream_sha256(path: Path, chunk_bytes=2**20) -> str:
    require(type(chunk_bytes) is int and chunk_bytes > 0,"hash分块非法")
    h=hashlib.sha256()
    with Path(path).open("rb") as f:
        while True:
            data=f.read(chunk_bytes)
            if not data:break
            h.update(data)
    return h.hexdigest()

def ensure_local_directory(path: str|Path) -> Path:
    raw=Path(path).expanduser()
    require(not raw.is_symlink(),"不能通过符号链接提供模型目录")
    require(raw.is_dir() and (raw/"config.json").is_file(),
            "必须是包含config.json的现有本地目录，不接受仓库ID或URL")
    folder=raw.resolve(strict=True)
    require(not folder.is_symlink(),"真实目录不能是符号链接")
    return folder

def read_limited_json(path: Path,max_bytes: int=2_000_000):
    require(path.is_file() and not path.is_symlink(),"JSON文件必须是普通文件，不允许符号链接")
    require(0<path.stat().st_size<=max_bytes,"JSON文件为空或超过安全读取上限")
    return strict_json(path.read_text(encoding="utf-8"))

def safetensors_header(path: Path) -> dict:
    """只读前缀，不通过torch.load打开权重；不是对任意恶意文件的安全认证。"""
    require(path.is_file() and not path.is_symlink(),"权重文件不能是符号链接")
    size=path.stat().st_size
    require(size>=16,"safetensors文件太短")
    with path.open("rb") as source:
        first=source.read(8)
        require(len(first)==8,"权重文件缺失header长度")
        n=struct.unpack("<Q",first)[0]
        require(2<=n<=16*2**20 and n+8<=size,"safetensors header非法或过大")
        head=source.read(n)
        require(len(head)==n,"safetensors header截断")
    header=strict_json(head.decode("utf-8")) if n<=8192 else json.loads(head.decode("utf-8"),
        object_pairs_hook=lambda pairs: _no_duplicate_object(pairs),
        parse_constant=lambda x: (_ for _ in ()).throw(LessonError("非法常量")))
    require(type(header) is dict and bool(header),"safetensors需要JSON对象header")
    data_len=size-8-n
    names=[]
    for key,value in header.items():
        if key=="__metadata__":
            require(type(value) is dict,"safetensors metadata需要对象")
            continue
        require(type(key) is str and 0<len(key)<=4096 and type(value) is dict,
                "safetensors tensor条目非法")
        require(type(value.get("dtype")) is str and type(value.get("shape")) is list,
                "safetensors dtype/shape缺失")
        require(all(type(dim) is int and dim>=0 for dim in value["shape"]),"tensor shape非法")
        offsets=value.get("data_offsets")
        require(type(offsets) is list and len(offsets)==2 and
                all(type(x) is int for x in offsets) and 0<=offsets[0]<=offsets[1]<=data_len,
                "tensor data offsets越界")
        names.append(key)
    require(bool(names),"safetensors文件没有tensor")
    return {"tensors":len(names),"header_sha256":hashlib.sha256(head).hexdigest(),
            "header_size":n,"file_size":size,"data_size":data_len,
            "tensor_names_digest":digest(sorted(names)),"_tensor_names":names}

def _no_duplicate_object(items):
    out={}
    for k,v in items:
        require(k not in out,"重复safetensors JSON字段")
        out[k]=v
    return out

def inspect_local_checkpoint(path: str|Path, *, full_weight_hash=False,
                             max_weight_bytes: int=2*1024**3) -> dict:
    """metadata仅辅助检查文件身份；默认并不声称哈希全部权重。"""
    require(type(full_weight_hash) is bool,"full_weight_hash必须是bool")
    require(type(max_weight_bytes) is int and max_weight_bytes>0,"资源上限需要正整数")
    folder=ensure_local_directory(path)
    cfg=read_limited_json(folder/"config.json")
    require(type(cfg) is dict and type(cfg.get("model_type")) is str,
            "config.json缺少model_type；不会猜测模型架构")
    require(cfg.get("is_encoder_decoder",False) is False,"仅支持decoder-only因果LM")
    shards=sorted(folder.glob("*.safetensors"))
    require(bool(shards),"没有safetensors；本课拒绝回退到pickle权重加载")
    require(all(p.is_file() and not p.is_symlink() for p in shards),"权重分片必须是普通文件")
    index_files=sorted(folder.glob("*.safetensors.index.json"))
    require(len(index_files)<=1,"多个safetensors索引无法确认完整权重")
    require(len(shards)==1 or bool(index_files),
            "多个safetensors分片需要明确的weight_map索引")
    if index_files:
        idx=read_limited_json(index_files[0],max_bytes=8*2**20)
        require(type(idx) is dict and type(idx.get("weight_map")) is dict and bool(idx["weight_map"]),
                "权重索引缺少weight_map")
        listed={str(v) for v in idx["weight_map"].values()}
        require(all(type(v) is str and v==Path(v).name and v.endswith(".safetensors") for v in listed),
                "索引权重文件不能穿越目录")
        require(listed=={p.name for p in shards},"索引和权重分片集合不一致")
    total=sum(p.stat().st_size for p in shards)
    require(total<=max_weight_bytes,
            "safetensors总字节超过显式本地资源闸门；没有尝试载入模型")
    infos=[]
    shard_names={}
    for p in shards:
        header=safetensors_header(p)
        shard_names[p.name]=set(header.pop("_tensor_names"))
        infos.append({"file":p.name,"bytes":p.stat().st_size,
                      "mtime_ns":p.stat().st_mtime_ns,"header":header})
    if index_files:
        mapping=idx["weight_map"]
        require(set(mapping)==set().union(*shard_names.values()),
                "weight_map与各safetensors header的tensor集合不匹配")
        require(all(tensor in shard_names[file_name] for tensor,file_name in mapping.items()),
                "weight_map把tensor映射到了错误的shard")
    if full_weight_hash:
        for part,p in zip(infos,shards):part["full_sha256"]=stream_sha256(p)
    small=[]
    for name in ("config.json","tokenizer.json","tokenizer_config.json","special_tokens_map.json",
                 "generation_config.json","chat_template.jinja","tokenizer.model","spiece.model",
                 "sentencepiece.bpe.model","vocab.json","vocab.txt","merges.txt","added_tokens.json",
                 *(p.name for p in index_files)):
        p=folder/name
        if p.exists():
            require(p.is_file() and not p.is_symlink(),"辅助文件不能是符号链接")
            require(p.stat().st_size<=64*2**20,"配置/tokenizer文件超出本课审计上限")
            small.append({"file":name,"bytes":p.stat().st_size,"sha256":stream_sha256(p),
                          "mtime_ns":p.stat().st_mtime_ns})
    require(bool(small) and small[0]["file"]=="config.json","必须审计config.json")
    identity_material={"shards":infos,"small_files":small,"model_type":cfg["model_type"]}
    return {"scope":"local_files_metadata_only" if not full_weight_hash else "local_full_weight_sha256",
            "directory_label":folder.name,"model_type":cfg["model_type"],
            "weight_file_count":len(shards),"weight_total_bytes":total,
            "weight_mib":round(total/2**20,3),
            "max_weight_bytes_gate":max_weight_bytes,
            "metadata_fingerprint":digest(identity_material),
            "full_weight_content_hashed":full_weight_hash,
            "cryptographic_weight_identity":full_weight_hash,
            "weight_files":infos,"small_files":small,
            "warning":"metadata+header fingerprint is not full tensor identity" if not full_weight_hash
                      else "full local weights hashed; still not a trust/security certificate"}

def validate_output_outside_model(model_dir: Path,output_root: str|Path):
    folder=ensure_local_directory(model_dir)
    out=Path(output_root).expanduser().resolve()
    require(out!=folder and folder not in out.parents,
            "报告目录不能位于模型目录内部，避免修改源checkpoint")

def inspect_local_training_candidate(path:str|Path,max_weight_bytes:int=2*1024**3,
                                     full_weight_hash=False) -> dict:
    result=inspect_local_checkpoint(path,full_weight_hash=full_weight_hash,
                                    max_weight_bytes=max_weight_bytes)
    folder=ensure_local_directory(path)
    config=read_limited_json(folder/'config.json')
    require(config.get('is_encoder_decoder',False) is False,'本课仅接受decoder-only')
    require(any((folder/name).is_file() and not (folder/name).is_symlink()
                for name in ('tokenizer.json','tokenizer.model','vocab.json','vocab.txt')),
            '没有本地tokenizer证据；不下载补齐')
    result['preflight']='metadata_only_does_not_load_or_train'
    result['needs_explicit_training_flag']=True
    result['warning']+='; local_files_only and trust_remote_code=False are not a sandbox'
    return result

def _hf_encode_example(tokenizer,row,max_length=160):
    """真实模型必须使用自身Chat Template，并逐token核对提示边界。"""
    require(getattr(tokenizer,'chat_template',None) is not None,
            '本地模型tokenizer缺少chat_template，不自动猜拼接规则')
    require(getattr(tokenizer,'eos_token_id',None) is not None,'EOS未定义')
    messages=messages_for_row(row)
    prefix=tokenizer.apply_chat_template(messages,tokenize=True,add_generation_prompt=True)
    full=tokenizer.apply_chat_template(messages+[{'role':'assistant','content':row['assistant']}],
                                       tokenize=True,add_generation_prompt=False)
    require(type(prefix) is list and type(full) is list and
            all(type(x) is int for x in full+prefix),'需要一维token ID列表')
    require(full[:len(prefix)]==prefix and len(full)>len(prefix),
            '真实模板的assistant header与完整训练序列token边界不一致，不能猜损失掩码')
    eos=tokenizer.eos_token_id
    require(type(eos) is int and full[-1]==eos,
            '真实模板不以tokenizer.eos_token_id结束；须先确定模型停止契约')
    require(len(full)<=max_length,'真实训练样本超过长度；本课不执行截断')
    require(len(full)-len(prefix)>1,'没有完整的回答加EOS')
    return {'input_ids':full,'labels':[IGNORE]*len(prefix)+full[len(prefix):],
            'prompt_len':len(prefix),'answer_len':len(full)-len(prefix)}

In [10]:
print('没有执行本地HF检查：没有提供受信任的本地路径。')
print('HF/PEFT是否已在当前环境可用:',
      {name:importlib.util.find_spec(name) is not None for name in ('transformers','peft','safetensors')})

没有执行本地HF检查：没有提供受信任的本地路径。
HF/PEFT是否已在当前环境可用: {'transformers': False, 'peft': False, 'safetensors': True}


## 5. 发布门槛：开发集给出决定，测试集保留未知

候选被独立恢复、评分通过后，才允许考虑发布。默认门槛：

\[
\text{NLL}_{candidate,dev}\le\text{NLL}_{baseline,dev}-10^{-4}.
\]

**必须拒绝 NaN、错误身份、错误词表、形状不对或分数声明与实际重算不一致的候选。** 门槛即使通过，也不意味着测试成功率必然变好。

本课强制两个版本：`v0000` 是初始零增量adapter，`v0001` 是开发集所选训练结果。开发门槛失败时，`v0001` 只留在staging，当前版本仍为 `v0000`。

开发集不等于测试集：若查看测试结果后重选步数，就不能再把该测试当作盲测。

In [11]:
@dataclass(frozen=True)
class ReleaseConfig:
    seed: int = 7001
    steps: int = 32
    batch_size: int = 8
    lr: float = 0.02
    eval_every: int = 8
    max_length: int = 160
    rank: int = 4
    alpha: float = 8.0
    max_grad_norm: float = 1.0
    min_dev_improvement: float = 0.0001
    max_static_mib: float = 64.0
    generation_tokens: int = 20

    def validate(self):
        require(type(self.seed) is int and self.seed >= 0, 'seed必须非负整数')
        for k in ('steps','batch_size','eval_every','max_length','rank','generation_tokens'):
            require(type(getattr(self,k)) is int and getattr(self,k)>0, k+'必须为正整数')
        for k in ('lr','alpha','max_grad_norm','max_static_mib'):
            v=getattr(self,k)
            require(type(v) in (int,float) and math.isfinite(v) and v>0,k+'需要有限正数')
        require(type(self.min_dev_improvement) in (int,float) and
                math.isfinite(self.min_dev_improvement) and self.min_dev_improvement>=0,
                'dev改善阈值需要有限非负')


def adapter_parameters(model: TinyChatLM) -> list[nn.Parameter]:
    params = [p for _,p in adapters(model)]
    require(bool(params) and all(p.requires_grad for p in params),'缺少可训练LoRA参数')
    require(all(p.requires_grad==n.endswith(('.A','.B')) for n,p in model.named_parameters()),
            '优化前只允许A/B为可训练')
    return params


@torch.no_grad()
def greedy_toy(model: TinyChatLM,tok: ByteMergeChatTokenizer,row: dict,max_new_tokens:int=20) -> dict:
    require(type(max_new_tokens) is int and max_new_tokens>0, '生成预算非法')
    prefix=tok.apply_chat_template(messages_for_row(row),tokenize=True,
                                    add_generation_prompt=True)
    require(len(prefix)<model.cfg.max_length,'prompt耗尽上下文')
    device=next(model.parameters()).device
    tokens=prefix[:];generated=[];old_mode=model.training
    model.eval();reason='max_tokens'
    try:
        for _ in range(max_new_tokens):
            if len(tokens)>=model.cfg.max_length:
                reason='context_limit';break
            ids=torch.tensor([tokens],dtype=torch.long,device=device)
            mask=torch.ones_like(ids,dtype=torch.bool)
            scores=model(ids,mask)[0,-1].clone()
            # 教学中的生成协议不允许控制符作为回答正文，EOS除外。
            for control in list(tok.specials):
                if control!=tok.eos_token_id:scores[control]=-torch.inf
            require(bool(torch.isfinite(scores[tok.eos_token_id])), 'EOS分数非法')
            nxt=int(scores.argmax());tokens.append(nxt);generated.append(nxt)
            if nxt==tok.eos_token_id:
                reason='eos';break
    finally:model.train(old_mode)
    body=generated[:-1] if reason=='eos' else generated
    try:
        text=tok.decode(body)
        valid_utf8=True
    except LessonError:
        # 真实模型可能产生未完成的UTF-8；保留原始token，不替换字符伪装正确。
        text=None;valid_utf8=False
    return {'id':row['id'],'expected':row['assistant'],'text':text,'valid_utf8':valid_utf8,
            'token_ids':generated,'stop_reason':reason,
            'strict_match':bool(valid_utf8 and reason=='eos' and text==row['assistant'])}


@torch.no_grad()
def generation_report(model,rows,tok,max_new_tokens=20):
    require(type(rows) is list and bool(rows),'没有生成评测样本')
    cases=[greedy_toy(model,tok,r,max_new_tokens) for r in rows]
    return {'cases':cases,'strict_successes':sum(c['strict_match'] for c in cases),
            'total':len(cases),'strict_success_rate':sum(c['strict_match'] for c in cases)/len(cases),
            'note':'这是实际贪心生成；teacher-forced NLL不是生成准确率'}


def train_toy_candidate(policy: TinyChatLM,base: TinyChatLM,tok: ByteMergeChatTokenizer,
                        data:dict,cfg:ReleaseConfig) -> dict:
    cfg.validate();validate_dataset(data)
    require(base_hash(policy)==state_hash(base.state_dict()),'训练前基座身份不一致')
    baseline=evaluate_nll(policy,data['dev'],tok,cfg.max_length)
    base_before=base_hash(policy);weights_before=state_hash(adapter_snapshot(policy))
    optimizer=torch.optim.AdamW(adapter_parameters(policy),lr=cfg.lr,weight_decay=0.0)
    rng=random.Random(cfg.seed+101)
    best_snapshot=adapter_snapshot(policy);best_step=0;best_loss=baseline['teacher_forced_nll']
    history=[{'step':0,'dev_nll':best_loss,'selected':True}]
    for step in range(1,cfg.steps+1):
        rows=rng.sample(data['train'],min(cfg.batch_size,len(data['train'])))
        result=toy_adapter_step(policy,optimizer,rows,tok,cfg.max_length,cfg.max_grad_norm)
        if step%cfg.eval_every==0 or step==cfg.steps:
            dev=evaluate_nll(policy,data['dev'],tok,cfg.max_length)
            improved=dev['teacher_forced_nll']<best_loss
            history.append({'step':step,'batch_nll':result['loss'],
                            'dev_nll':dev['teacher_forced_nll'],'selected':improved})
            if improved:
                best_loss=dev['teacher_forced_nll'];best_step=step
                best_snapshot=adapter_snapshot(policy)
    restore_adapter(policy,best_snapshot)
    policy.eval()
    require(base_hash(policy)==base_before==state_hash(base.state_dict()),'候选训练修改了基座')
    require(state_hash(adapter_snapshot(policy))!=weights_before or best_step==0,
            '训练没有改变adapter却报告成功')
    require(all(p.grad is None for p in base.parameters()),'基座异常积累梯度')
    return {'selected_step':best_step,'baseline_dev_nll':baseline['teacher_forced_nll'],
            'selected_dev_nll':best_loss,'history':history,
            'rule':'minimum dev token NLL; include step 0; earliest tie; test unseen',
            'base_unchanged':True,'all_trainable_lora':True}

In [12]:
print('简短的真实生成:',greedy_toy(small_policy,tok,data['dev'][0],max_new_tokens=5))

简短的真实生成: {'id': 's02b00', 'expected': '{"answer":"00"}', 'text': None, 'valid_utf8': False, 'token_ids': [99, 217, 222, 152, 60], 'stop_reason': 'max_tokens', 'strict_match': False}


In [13]:
TRAIN_CFG = ReleaseConfig(seed=SEED,rank=RANK,alpha=ALPHA)
print('完整默认训练配置:',asdict(TRAIN_CFG))

完整默认训练配置: {'seed': 7001, 'steps': 32, 'batch_size': 8, 'lr': 0.02, 'eval_every': 8, 'max_length': 160, 'rank': 4, 'alpha': 8.0, 'max_grad_norm': 1.0, 'min_dev_improvement': 0.0001, 'max_static_mib': 64.0, 'generation_tokens': 20}


In [14]:
print('候选门槛示例：')
for candidate,minimum in [(3.8,0.0001),(5.9,.1),(6.1,0.)]:
    print('初始6.0，候选',candidate,'最小改善',minimum,
          '→',candidate<=6.0-minimum)

候选门槛示例：
初始6.0，候选 3.8 最小改善 0.0001 → True
初始6.0，候选 5.9 最小改善 0.1 → True
初始6.0，候选 6.1 最小改善 0.0 → False


## 6. Adapter-only 存储协议：先stage，再commit

```
某次运行/
  base_for_replay.pt              # 仅教学随机基座的复现文件，不属于adapter版本
  versions/
    v0000/adapter.pt, manifest.json
    v0001/adapter.pt, manifest.json
  .staging/
    v0002/adapter.pt, manifest.json     # 未发布或被拒绝的候选
  VERSION_HASHES.json             # 已发布版本manifest摘要登记表
  CURRENT.json                    # 当前版本指针
  audit.jsonl                     # 发布、回滚、恢复历史
  report.json / recovery.json
```

**关键区别：** `versions/vNNNN/adapter.pt` 只保存LoRA A/B，不包含基座。由于随机Toy基座不在外部仓库，本课另在顶层保存一次 `base_for_replay.pt` 以便独立进程恢复；这不是官方PEFT格式。真正HF可选模式则保存 `adapter_model.safetensors`＋`adapter_config.json`，仍要求用户持有原始本地模型。

仓库在stage阶段执行：权重与manifest哈希、身份与shape校验、全新策略独立加载、重新计算开发集loss。失败不允许修改CURRENT。

发布时先将staging目录同文件系统rename为不可覆盖的version，再登记manifest摘要，最后写临时指针、fsync并 `os.replace` 到CURRENT。**版本发布和CURRENT激活是两个可观察事件。**

In [15]:
SAFE_VERSION=re.compile(r'^v[0-9]{4}$')


def _fsync_directory(folder:Path) -> bool:
    """POSIX本地文件系统尽力flush目录；不是跨平台/网络FS的崩溃持久性证明。"""
    try:
        fd=os.open(str(folder),os.O_RDONLY)
    except OSError:
        return False
    try:
        os.fsync(fd)
        return True
    except OSError:
        return False
    finally:
        os.close(fd)


def atomic_json(path:Path,value) -> None:
    """同目录先写全量临时文件+fsync，再os.replace发布。单写者约定。"""
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    require(not path.is_symlink(),'原有指针不能是符号链接')
    tmp=path.with_name(path.name+'.tmp.'+str(os.getpid()))
    require(not tmp.exists(),'临时发布文件已存在，拒绝覆盖')
    try:
        with tmp.open('x',encoding='utf-8') as f:
            f.write(json.dumps(value,sort_keys=True,ensure_ascii=False,indent=2,allow_nan=False)+'\n')
            f.flush();os.fsync(f.fileno())
        os.replace(tmp,path)
        _fsync_directory(path.parent)
    finally:
        if tmp.exists():tmp.unlink()


def safe_file(path:Path,max_size:int=64*2**20) -> Path:
    require(path.is_file() and not path.is_symlink(),'文件不存在或为符号链接')
    require(0<path.stat().st_size<=max_size,'文件为空/超过本课大小上限')
    return path


def compact_file_hash(path:Path) -> str:
    return stream_sha256(safe_file(path))


def read_json_bounded(path:Path, max_bytes:int=1024*1024):
    """运行报告可能超过8KiB；仍限制总字节、重复键和NaN。"""
    path=safe_file(path,max_bytes)
    def reject_constant(value):raise LessonError('JSON包含NaN或Infinity')
    try:
        return json.loads(path.read_text(encoding='utf-8'),
                          object_pairs_hook=_no_duplicate_object,parse_constant=reject_constant)
    except (json.JSONDecodeError,UnicodeError,RecursionError) as exc:
        raise LessonError('运行JSON解析失败') from exc


def adapter_identity(model:TinyChatLM) -> dict:
    layers=[m for m in model.modules() if isinstance(m,LoRALinear)]
    require(bool(layers),'没有LoRA结构')
    require(len({(m.rank,m.alpha) for m in layers})==1,'混用的LoRA配置本课不支持')
    return {'rank':layers[0].rank,'alpha':layers[0].alpha,
            'target_count':len(layers),'name_shapes':{
                name:list(p.shape) for name,p in adapters(model)}}


class ToyReleaseStore:
    """仅用于可信本地toy权重。version目录只创建，CURRENT是单独的可回滚指针。"""
    def __init__(self,root:Path,base:TinyChatLM,tok:ByteMergeChatTokenizer,
                 data:dict,rank:int,alpha:float,max_length:int):
        self.root=Path(root)
        require(not self.root.is_symlink(),'不能把发布仓库设为符号链接')
        self.root.mkdir(parents=True,exist_ok=True)
        self.root=self.root.resolve()
        self.versions=self.root/'versions';self.staging=self.root/'.staging'
        for folder in (self.versions,self.staging):
            require(not folder.is_symlink(),'版本目录不能是符号链接')
            folder.mkdir(exist_ok=True)
        self.pointer=self.root/'CURRENT.json';self.audit=self.root/'audit.jsonl'
        require(not self.pointer.is_symlink(),'发布指针不能是符号链接')
        self.base=base;self.tok=tok;self.data=data;self.rank=rank;self.alpha=alpha
        self.max_length=max_length
        self.base_id=state_hash(base.state_dict());self.data_id=digest(data)
        self.tok_id=digest(tok.contract());self.model_cfg=asdict(base.cfg)
        validate_dataset(data)
        require(not any(p.requires_grad for p in base.parameters()),'发布源基座必须冻结')
        require(type(rank)is int and rank>0 and alpha>0,'LoRA配置非法')

    def _version_folder(self,version:str,published:bool)->Path:
        require(type(version)is str and SAFE_VERSION.fullmatch(version),'版本ID必须是v后4位数字')
        root=self.versions if published else self.staging
        folder=root/version
        require(not folder.is_symlink(),'版本目录不允许符号链接')
        return folder

    def _registry(self):
        index=self.root/'VERSION_HASHES.json'
        if not index.exists():return {}
        loaded=read_json_bounded(index,128*1024)
        require(type(loaded) is dict and all(
            type(k)is str and SAFE_VERSION.fullmatch(k) and
            type(v)is str and re.fullmatch(r'[0-9a-f]{64}',v)
            for k,v in loaded.items()),'发布版本哈希索引损坏')
        return loaded

    def _register(self,version):
        registry=self._registry()
        require(version not in registry,'同一版本不能重新登记')
        folder=self._version_folder(version,True)
        registry[version]=compact_file_hash(folder/'manifest.json')
        atomic_json(self.root/'VERSION_HASHES.json',registry)

    def _manifest(self,version,policy,dev_nll,metadata,bytes_hash):
        require(type(metadata)is dict,'metadata必须是JSON对象')
        return {'version_format':VERSION,'backend':'toy','version':version,
                'base_hash':self.base_id,'data_hash':self.data_id,
                'tokenizer_contract_hash':self.tok_id,'model_config':self.model_cfg,
                'adapter_spec':adapter_identity(policy),'tensor_sha256':state_hash(adapter_snapshot(policy)),
                'file_sha256':bytes_hash,'dev_nll':float(dev_nll),
                'selection_metadata':metadata,'format':'trusted_local_torch_weights_only'}

    def stage(self,version:str,policy:TinyChatLM,metadata:dict) -> Path:
        require(base_hash(policy)==self.base_id,'staging adapter与基座错配')
        spec=adapter_identity(policy)
        require(spec['rank']==self.rank and spec['alpha']==self.alpha,
                'staging LoRA rank/alpha与发布仓库不一致')
        folder=self._version_folder(version,False)
        require(not folder.exists() and not self._version_folder(version,True).exists(),
                '该版本在staging或versions中已存在；拒绝覆盖')
        folder.mkdir()
        try:
            dev=evaluate_nll(policy,self.data['dev'],self.tok,self.max_length)
            state=adapter_snapshot(policy)
            torch.save({'format':VERSION,'kind':'toy_adapter','state':state},folder/'adapter.pt')
            file_hash=compact_file_hash(folder/'adapter.pt')
            manifest=self._manifest(version,policy,dev['teacher_forced_nll'],metadata,file_hash)
            write_json(folder/'manifest.json',manifest)
            with (folder/'adapter.pt').open('rb') as file:
                os.fsync(file.fileno())
            with (folder/'manifest.json').open('rb') as file:
                os.fsync(file.fileno())
            _fsync_directory(folder)
        except Exception:
            # 保留staging内容供人工审计，不默认删除失败工件。
            raise
        return folder

    def validate(self,version:str,published:bool=False) -> tuple[TinyChatLM,dict]:
        folder=self._version_folder(version,published)
        require(folder.is_dir(),'版本不存在')
        if published:
            expected=self._registry().get(version)
            require(expected is not None and
                    compact_file_hash(folder/'manifest.json')==expected,
                    '已发布版本不在登记表或manifest字节发生变化')
        require({p.name for p in folder.iterdir()}=={'adapter.pt','manifest.json'},
                '版本内容必须恰好两个文件，不允许混入链接/旧优化器状态')
        man=read_limited_json(safe_file(folder/'manifest.json',max_size=256*1024),256*1024)
        require(type(man) is dict and set(man)=={'version_format','backend','version','base_hash',
                'data_hash','tokenizer_contract_hash','model_config','adapter_spec','tensor_sha256',
                'file_sha256','dev_nll','selection_metadata','format'},'manifest字段不匹配')
        require(man['version_format']==VERSION and man['backend']=='toy' and man['version']==version,
                'manifest版本身份不符')
        require(man['base_hash']==self.base_id and man['data_hash']==self.data_id and
                man['tokenizer_contract_hash']==self.tok_id and man['model_config']==self.model_cfg,
                '基座/模板/数据/模型配置不匹配')
        require(man['format']=='trusted_local_torch_weights_only','只接受本课受信任的自生成adapter')
        require(type(man['dev_nll']) in (int,float) and math.isfinite(man['dev_nll']) and
                man['dev_nll']>=0,'dev_nll非法')
        require(type(man['selection_metadata']) is dict,'选择说明错误')
        require(compact_file_hash(folder/'adapter.pt')==man['file_sha256'],'adapter文件摘要失效')
        # weights_only仍不等价于“任意不可信文件可安全加载”：只能加载本课自生成工件。
        blob=torch.load(folder/'adapter.pt',map_location='cpu',weights_only=True)
        require(type(blob)is dict and set(blob)=={'format','kind','state'} and
                blob['format']==VERSION and blob['kind']=='toy_adapter','adapter载荷不符合协议')
        require(type(blob['state']) is dict and state_hash(blob['state'])==man['tensor_sha256'],
                'adapter张量身份改变')
        policy=inject_lora(self.base,self.rank,self.alpha)
        require(adapter_identity(policy)==man['adapter_spec'],'adapter名称/形状/LoRA设置错配')
        restore_adapter(policy,blob['state'])
        policy.requires_grad_(False).eval()
        require(base_hash(policy)==self.base_id,'重载改变基座')
        computed=evaluate_nll(policy,self.data['dev'],self.tok,self.max_length)
        require(abs(computed['teacher_forced_nll']-man['dev_nll'])<2e-5,
                '重新评分与发布承诺不一致，拒绝发布/切换')
        return policy,man

    def _write_pointer(self,version:str) -> None:
        folder=self._version_folder(version,True)
        require(folder.is_dir(),'不能指向未发布版本')
        manifest_hash=compact_file_hash(folder/'manifest.json')
        require(self._registry().get(version)==manifest_hash,
                'manifest未登记或发布后发生变化')
        atomic_json(self.pointer,{'format':VERSION,'backend':'toy','version':version,
                                 'manifest_sha256':manifest_hash})

    def current(self):
        require(self.pointer.exists(),'尚未发布任何版本')
        pointer=read_limited_json(safe_file(self.pointer,16*1024),16*1024)
        require(type(pointer)is dict and set(pointer)=={'format','backend','version','manifest_sha256'}
                and pointer['format']==VERSION and pointer['backend']=='toy','CURRENT格式错误')
        version=pointer['version'];folder=self._version_folder(version,True)
        require(compact_file_hash(folder/'manifest.json')==pointer['manifest_sha256'],
                'CURRENT指向的manifest被改动')
        policy,man=self.validate(version,True)
        return version,policy,man

    def _log(self,action,version,details):
        require(not self.audit.is_symlink(),'审计日志不能是符号链接')
        with self.audit.open('a',encoding='utf-8') as handle:
            handle.write(canonical({'action':action,'version':version,'details':details})+'\n')
            handle.flush();os.fsync(handle.fileno())

    def promote(self,version:str,baseline_dev:float,min_improvement:float,
                *,fault:str|None=None) -> dict:
        require(type(fault) in (str,type(None)) and fault in (None,'before_rename','before_pointer'),
                '故障位置非法')
        require(type(baseline_dev) in (int,float) and math.isfinite(baseline_dev) and
                baseline_dev>=0 and math.isfinite(min_improvement) and min_improvement>=0,
                '开发阈值非法')
        policy,man=self.validate(version,False)
        if version!='v0000':
            require(man['dev_nll']<=baseline_dev-min_improvement,
                    '候选没有达到预先规定的开发集门槛；拒绝发布')
        if version=='v0000':
            require(not self.pointer.exists(),'初始版本只能发布一次')
        else:
            self.current()  # 发布前已有可信当前版本；失败不得清空指针。
        if fault=='before_rename':raise LessonError('模拟失败：版本目录发布前')
        stage=self._version_folder(version,False);dest=self._version_folder(version,True)
        require(not dest.exists(),'目标版本已存在，绝不覆盖')
        os.replace(stage,dest)  # 同一文件系统目录rename；不替换已存在的版本目录。
        _fsync_directory(self.versions)
        # 先登记不可变版本的文件摘要，再原子切换活动指针。
        # 若中断，已登记版本可见但不会自动成为CURRENT。
        self._register(version)
        if fault=='before_pointer':
            raise LessonError('模拟失败：版本已可见，但指针未更新；属于未激活孤儿版本')
        self._write_pointer(version)
        self._log('promote',version,{'dev_nll':man['dev_nll']})
        return {'version':version,'published':True,'dev_nll':man['dev_nll'],
                'loaded_parameters_trainable':sum(p.numel() for p in policy.parameters() if p.requires_grad)}

    def switch_to(self,version:str,reason:str='rollback') -> dict:
        require(reason in {'rollback','restore','manual_selection'},'操作原因不合法')
        old=self.current()[0]
        policy,man=self.validate(version,True)
        self._write_pointer(version)
        self._log(reason,version,{'from':old,'dev_nll':man['dev_nll']})
        return {'from':old,'to':version,'action':reason,'trainable':sum(p.numel() for p in policy.parameters() if p.requires_grad)}

## 7. 为什么回滚应该切换指针，而不是删除权重？

假设 CURRENT 是 `v0001`，发现问题想回到 `v0000`：

```
CURRENT v0001  →  CURRENT v0000
```

两份已发布权重仍然在 `versions/` 中。只有当目标版本存在、登记摘要完整、base/tokenizer身份匹配、**全新对象独立加载和评分通过**时，才切换指针。恢复候选也是相同验证后的指针更新。

版本目录rename之后、指针更新之前发生错误，会留下“未激活版本”；它不等于“当前线上版本”。本课故障注入检验这一点。

单机本地同一文件系统上的 `os.replace` 可帮助获得单次指针替换的原子可见性；**没有跨目录、多写者、网络文件系统事务保证，也不声称强制磁盘不掉电。**

In [16]:
with tempfile.TemporaryDirectory() as tmp:
    path=Path(tmp)/'CURRENT.json'
    atomic_json(path, {'version':'v0000','state':'baseline'})
    print('第一次指针:',read_json_bounded(path))
    atomic_json(path, {'version':'v0001','state':'candidate'})
    print('激活候选:',read_json_bounded(path))
    atomic_json(path, {'version':'v0000','state':'rollback'})
    print('回滚:',read_json_bounded(path))
    print('临时指针残留:',list(Path(tmp).glob('CURRENT.json.tmp*')))

第一次指针: {'state': 'baseline', 'version': 'v0000'}
激活候选: {'state': 'candidate', 'version': 'v0001'}
回滚: {'state': 'rollback', 'version': 'v0000'}
临时指针残留: []


## 8. 默认完整实验：实际训练、门槛、回滚与最终测试

下面的默认训练操作一次性完成：

1. 固定随机基座与合成数据，先创建并发布 `v0000`；
2. 只用训练数据更新A/B，开发集选择候选；
3. stage到 `v0001`，重建独立策略、核对开发指标；
4. 通过门槛才发布，回滚到 `v0000`、校验logits，再恢复 `v0001`；
5. 故意把一个新staging版本的基座身份写坏，确认CURRENT不变；
6. **这些全部结束后才首次读取测试集**，同时报告teacher-forced NLL与真实生成准确率。

全部结果由下面代码真实计算；若显式关闭 `RUN_FULL_TRAINING`，不制造训练成绩。

In [17]:
def new_run_dir(root:Path):
    root=Path(root)
    require(not root.is_symlink(),'输出根目录不能是符号链接')
    root.mkdir(parents=True,exist_ok=True)
    run=root/('run_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))
    run.mkdir(exist_ok=False)
    return run


def run_toy_release(output_root='lesson_70_outputs',device_name='cpu',
                    cfg:ReleaseConfig|None=None)->tuple[Path,dict]:
    cfg=cfg or ReleaseConfig();cfg.validate();device=device_from_name(device_name)
    torch.set_num_threads(1);seed_all(cfg.seed)
    data=make_dataset(cfg.seed);validate_dataset(data)
    tok=ByteMergeChatTokenizer()
    base=TinyChatLM(ToyConfig(tok.vocab_size,max_length=cfg.max_length)).to(device)
    base.requires_grad_(False).eval()
    base_id=state_hash(base.state_dict())
    policy=inject_lora(base,cfg.rank,cfg.alpha)
    estimate=memory_accounting(policy,cfg.batch_size,cfg.max_length)
    require(estimate['bookkept_static_mib']<=cfg.max_static_mib,
            '静态记账超过明确预算；未执行训练')
    baseline=evaluate_nll(policy,data['dev'],tok,cfg.max_length)
    baseline_train=evaluate_nll(policy,data['train'],tok,cfg.max_length)
    # 任何test标签均未被训练或开发集选择使用。
    folder=new_run_dir(Path(output_root))
    # 教学随机基座无法从Hub重建；只在独立文件保存一次确切权重供恢复。
    # 版本目录中的adapter.pt始终只包含A/B，不把完整基座塞进adapter。
    torch.save({'version':VERSION,'config':asdict(base.cfg),
                'tokenizer':tok.contract(),'base_hash':base_id,
                'state':{n:v.detach().cpu().clone() for n,v in base.state_dict().items()}},
               folder/'base_for_replay.pt')
    store=ToyReleaseStore(folder,base,tok,data,cfg.rank,cfg.alpha,cfg.max_length)
    store.stage('v0000',policy,{'purpose':'initial zero-delta adapter',
                                'selection':'fixed initial baseline'})
    store.promote('v0000',baseline['teacher_forced_nll'],cfg.min_dev_improvement)
    initial_v,initial_p,_=store.current()
    require(initial_v=='v0000','初始发布版本错误')
    baseline_gen_dev=generation_report(initial_p,data['dev'],tok,cfg.generation_tokens)
    selection=train_toy_candidate(policy,base,tok,data,cfg)
    require(base_hash(policy)==base_id,'候选修改基座')
    stage_meta={'purpose':'training candidate selected on dev only',
                'selected_step':selection['selected_step'],
                'dev_before':selection['baseline_dev_nll'],
                'dev_selected':selection['selected_dev_nll'],
                'train_config':asdict(cfg)}
    store.stage('v0001',policy,stage_meta)
    staged_policy,_=store.validate('v0001',False)
    probe=collate_training_rows([encode_training_row(tok,r,cfg.max_length)
                                 for r in data['dev'][:2]],tok,device)
    with torch.no_grad():
        original=policy(probe['input_ids'],probe['attention_mask'])
        reloaded=staged_policy(probe['input_ids'],probe['attention_mask'])
    require(torch.allclose(original,reloaded,atol=1e-6,rtol=1e-5),
            '保存后独立对象输出不一致；拒绝发布')
    gate={'baseline_dev_nll':baseline['teacher_forced_nll'],
          'candidate_dev_nll':selection['selected_dev_nll'],
          'minimum_improvement':cfg.min_dev_improvement,
          'passed':selection['selected_step']>0 and
                   selection['selected_dev_nll']<=baseline['teacher_forced_nll']-cfg.min_dev_improvement}
    if gate['passed']:
        published=store.promote('v0001',baseline['teacher_forced_nll'],cfg.min_dev_improvement)
        current_v,published_policy,_=store.current()
        require(current_v=='v0001','发布候选后CURRENT没有更新')
        trained_gen_dev=generation_report(published_policy,data['dev'],tok,cfg.generation_tokens)
        # 回滚演练只切换已有且通过重载验证的版本；不删除候选权重。
        rollback=store.switch_to('v0000','rollback')
        rollback_v,rollback_policy,_=store.current()
        require(rollback_v=='v0000','回滚指针错误')
        with torch.no_grad():
            r0=rollback_policy(probe['input_ids'],probe['attention_mask'])
            initial=initial_p(probe['input_ids'],probe['attention_mask'])
        require(torch.equal(r0,initial),'回滚后logits并非原始版本')
        restore=store.switch_to('v0001','restore')
        require(store.current()[0]=='v0001','演练后未恢复候选版本')
        rollback_report={'performed':True,'rollback':rollback,'restore':restore,
                         'rollback_logits_bitwise_equal':True,
                         'candidate_preserved':(store.versions/'v0001').is_dir()}
    else:
        published={'version':'v0000','published':False,
                   'note':'候选未达到开发门槛；保留staging且不改变当前版本'}
        trained_gen_dev=None
        rollback_report={'performed':False,'reason':'candidate_not_published'}
    # 故障演练：坏的manifest不能改变CURRENT。
    active_before=store.current()[0]
    failed=0
    bad=store.stage('v0002',policy,{'purpose':'intentional rejected staging; do not publish'})
    manifest_path=bad/'manifest.json'
    broken=read_limited_json(manifest_path)
    broken['base_hash']='0'*64
    write_json(manifest_path,broken)
    try:store.promote('v0002',baseline['teacher_forced_nll'],cfg.min_dev_improvement)
    except LessonError:failed+=1
    require(store.current()[0]==active_before,'失败候选错误地改变了CURRENT')
    final_version,final_policy,final_manifest=store.current()
    # 在全部训练/门槛/发布/演练完成后才打开test。
    test_base=evaluate_nll(initial_p,data['test'],tok,cfg.max_length)
    test_final=evaluate_nll(final_policy,data['test'],tok,cfg.max_length)
    gen_before=generation_report(initial_p,data['test'],tok,cfg.generation_tokens)
    gen_after=generation_report(final_policy,data['test'],tok,cfg.generation_tokens)
    require(state_hash(base.state_dict())==base_id,'发布/评分过程中基座发生变化')
    require(all(p.grad is None for p in base.parameters()),'冻结基座出现梯度')
    report={'version':VERSION,'environment':{'python':platform.python_version(),
                 'torch':str(torch.__version__),'device':str(device),'seed':cfg.seed},
            'config':asdict(cfg),'base_hash':base_id,'data_hash':digest(data),
            'tokenizer_hash':digest(tok.contract()),
            'base_parameters':sum(p.numel() for p in base.parameters()),
            'adapter_parameters':sum(p.numel() for _,p in adapters(policy)),
            'static_memory':estimate,'train_baseline':baseline_train,
            'baseline_dev':baseline,'selection':selection,'gate':gate,
            'staged_vs_in_memory_logit_abs_error':float((original-reloaded).abs().max()),
            'promotion':published,'rollback_drill':rollback_report,
            'failure_staging_detected':failed,'active_version':final_version,
            'dev_generation_initial':baseline_gen_dev,
            'dev_generation_candidate':trained_gen_dev,
            'test_initial_nll':test_base,'test_final_nll':test_final,
            'test_initial_generation':gen_before,'test_final_generation':gen_after,
            'trainable_after_recovery':sum(p.numel() for p in final_policy.parameters() if p.requires_grad),
            'selection_contract':'dev-only selection/gate; test first opened after CURRENT finalized',
            'limits':['toy_random_base_not_pretrained','synthetic_copy_task',
                'teacher_forced_NLL_not_generation_accuracy','single_seed',
                'manual_toy_LoRA_not_a_PEFT_adapter_export','not_full_training_resume',
                'transaction_single_writer_local_fs_not_crashproof','no_CUDA_measurement_on_CPU']}
    write_json(folder/'data.json',data);write_json(folder/'report.json',report)
    recovery=reopen_toy_run(folder,device_name)
    write_json(folder/'recovery.json',recovery)
    print('base params:',report['base_parameters'],'adapter params:',report['adapter_parameters'])
    print('dev teacher-forced NLL:',round(baseline['teacher_forced_nll'],5),'→',
          round(selection['selected_dev_nll'],5),'gate:',gate['passed'])
    print('CURRENT:',final_version,'rollback tested:',rollback_report['performed'])
    print('test teacher-forced NLL:',round(test_base['teacher_forced_nll'],5),'→',
          round(test_final['teacher_forced_nll'],5))
    print('test strict greedy:',gen_before['strict_successes'],'→',gen_after['strict_successes'],
          '/',gen_after['total'])
    print('output:',folder)
    return folder,report


def reopen_toy_run(folder:Path,device_name='cpu') -> dict:
    """新进程可执行；不复用训练对象，也不修改文件。"""
    folder=Path(folder);device=device_from_name(device_name)
    data=read_json_bounded(folder/'data.json',512*1024)
    validate_dataset(data)
    report=read_json_bounded(folder/'report.json',512*1024)
    # 只给toy demo允许读取自身运行时保存的初始基座；adapter版本永不包含基座。
    base_blob=torch.load(folder/'base_for_replay.pt',map_location='cpu',weights_only=True)
    tok=ByteMergeChatTokenizer()
    require(base_blob['version']==VERSION and base_blob['tokenizer']==tok.contract(),
            'base_for_replay版本/分词约定不符')
    cfg=ToyConfig(**base_blob['config'])
    base=TinyChatLM(cfg).to(device)
    base.load_state_dict(base_blob['state'],strict=True)
    base.requires_grad_(False).eval()
    require(state_hash(base.state_dict())==base_blob['base_hash']==report['base_hash'],
            '精确基座不匹配')
    spec=report['config']
    store=ToyReleaseStore(folder,base,tok,data,spec['rank'],spec['alpha'],spec['max_length'])
    version,policy,_=store.current()
    require(version==report['active_version'],'CURRENT与已发布实验不匹配')
    eval_nll=evaluate_nll(policy,data['test'],tok,spec['max_length'])
    regen=generation_report(policy,data['test'],tok,spec['generation_tokens'])
    error=abs(eval_nll['teacher_forced_nll']-report['test_final_nll']['teacher_forced_nll'])
    require(error<2e-5 and regen==report['test_final_generation'],
            '恢复模型的测试指标/生成不同')
    return {'current_version':version,'test_nll_diff':error,
            'test_generation_equal':True,'all_inference_frozen':
                all(not p.requires_grad for p in policy.parameters()),
            'base_hash_equal':state_hash(base.state_dict())==report['base_hash']}

In [18]:
RELEASE_DIR=None
RELEASE_REPORT=None
if RUN_FULL_TRAINING:
    RELEASE_DIR,RELEASE_REPORT=run_toy_release(OUTPUT_ROOT,DEVICE,TRAIN_CFG)
else:
    print('已明确跳过完整训练；没有假造实验或发布结果。')

base params: 40000 adapter params: 1024
dev teacher-forced NLL: 5.96678 → 3.80353 gate: True
CURRENT: v0001 rollback tested: True
test teacher-forced NLL: 5.96661 → 3.80769
test strict greedy: 0 → 2 / 8
output: lesson_70_outputs/run_20261104T061202_103333Z


## 9. 如何检查门槛与最终指标？

开发集NLL下降，只表示给定正确答案前缀的平均损失降低；是否能自由生成符合 JSON 规则的回答，要看真正的生成记录。

在本课程制作环境的默认 CPU 运行里，开发集 NLL 从约5.967降到3.804，门槛通过；测试侧从约5.967降到3.808，严格贪心生成从0/8到2/8。**即使改善，绝对生成水平仍很低**，它仍是随机小模型的合成格式实验，不是模型实际产品质量或通用能力证据。

`v0000` 始终可用；回滚演练不会删除 `v0001`。报告中另外记录 `staged_vs_in_memory_logit_abs_error` 和独立进程重载结果。

In [19]:
if RELEASE_REPORT is not None:
    print('发布门槛:',RELEASE_REPORT['gate'])
    print('开发选定step:',RELEASE_REPORT['selection']['selected_step'])
    print('初始test NLL:',RELEASE_REPORT['test_initial_nll']['teacher_forced_nll'])
    print('最终test NLL:',RELEASE_REPORT['test_final_nll']['teacher_forced_nll'])
    print('真实生成:',RELEASE_REPORT['test_initial_generation']['strict_successes'],
          '→',RELEASE_REPORT['test_final_generation']['strict_successes'],'/',
          RELEASE_REPORT['test_final_generation']['total'])
    print('最终CURRENT:',RELEASE_REPORT['active_version'])

发布门槛: {'baseline_dev_nll': 5.9667848859514505, 'candidate_dev_nll': 3.8035267421177457, 'minimum_improvement': 0.0001, 'passed': True}
开发选定step: 32
初始test NLL: 5.966606685093471
最终test NLL: 3.8076869419642856
真实生成: 0 → 2 / 8
最终CURRENT: v0001


In [20]:
if RELEASE_REPORT is not None:
    for original,selected in zip(RELEASE_REPORT['test_initial_generation']['cases'][:3],
                                 RELEASE_REPORT['test_final_generation']['cases'][:3]):
        print('ID:',original['id'],'expected:',original['expected'])
        print('base:',repr(original['text']),original['stop_reason'],original['strict_match'])
        print('adapter:',repr(selected['text']),selected['stop_reason'],selected['strict_match'])
        print()

ID: s01b00 expected: {"answer":"00"}
base: None max_tokens False
adapter: '{"answer":"00"}' eos True

ID: s01b01 expected: {"answer":"01"}
base: None max_tokens False
adapter: '{"answer":"00"}' eos False

ID: s01b02 expected: {"answer":"02"}
base: None max_tokens False
adapter: '{"answer":"00"}' eos False



## 10. 重启恢复：不允许悄悄以最新目录代替CURRENT

恢复流程必须先检查 CURRENT 指向的版本及其摘要、发布登记表、完整基座身份、adapter schema和每一份张量，然后重建独立模型。

如果CURRENT损坏或指向不存在的版本，明确报错，**不能默默选择磁盘上名称最大的vNNNN**；名字最大不等于最后一次成功激活的版本。

下面从文件恢复推理模型，检查同一测试集的数值和生成记录。重载文件只能恢复推理：没有优化器、随机数状态和数据游标，不是精确续训。

In [21]:
if RELEASE_DIR is not None:
    replay=reopen_toy_run(RELEASE_DIR,DEVICE)
    print('重启恢复结果:',replay)
    print('CURRENT原始文件:',(RELEASE_DIR/'CURRENT.json').read_text(encoding='utf-8'))

重启恢复结果: {'current_version': 'v0001', 'test_nll_diff': 0.0, 'test_generation_equal': True, 'all_inference_frozen': True, 'base_hash_equal': True}
CURRENT原始文件: {
  "backend": "toy",
  "format": "lesson70-adapter-release-v1",
  "manifest_sha256": "b684737083d39faacce7e6f42a0f7f05b384405033a3d174f1e18e43ff630b55",
  "version": "v0001"
}



## 11. 显式真实本地 PEFT：使用本地tokenizer/template与safetensors

**仅当你已有可信的本地 decoder-only HF 模型，且提前安装了匹配版本的 `transformers`、`peft`、`safetensors` 时执行。** 当前教程的制作环境没有可加载的本地HF预训练权重及完整依赖，故下面只定义函数，不执行、不虚构HF或3060结果。

该路径要求：

- 用户明确传入现存的本地模型目录，检查 `config.json`、Safetensors分片、索引、tokenizer文件，默认权重大小上限256MiB；只支持 `nn.Linear` 的目标层，默认 `q_proj,v_proj`，非通用架构自动识别器。
- `from_pretrained(...local_files_only=True,trust_remote_code=False,use_safetensors=True)`，没有网络下载，也不自动安装依赖。以上仍不是不可信模型的安全沙盒。
- 使用真实tokenizer的 `apply_chat_template()` 和EOS约定，验证输入前缀/标签边界。默认只做少量更新，冻结基座，比较开发集NLL。
- `get_peft_model()`训练，`save_pretrained(...safe_serialization=True,save_embedding_layers=False)`生成 **真正PEFT适配器**，独立从原始基座调用 `PeftModel.from_pretrained(...is_trainable=False)`恢复，重新评分、查文件哈希后才能发布。
- 在真实本地模型运行时，首次打开测试仍应晚于开发集选择、版本提交及回滚；不会以train/dev NLL取代生成能力评测。

参考：[PEFT 的模型加载/保存契约](https://huggingface.co/docs/peft/main/package_reference/peft_model) 与 [LoRA配置](https://huggingface.co/docs/peft/main/package_reference/lora)。这些接口可能随已安装版本变化；失败应明确报告，不能隐式回退为Toy成功。

In [22]:
def _hf_token_batch(tokenizer,encoded:list[dict],pad_id:int,device):
    require(bool(encoded),'HF批次不能为空')
    width=max(len(row['input_ids']) for row in encoded)
    ids=torch.tensor([r['input_ids']+[pad_id]*(width-len(r['input_ids'])) for r in encoded],
                     dtype=torch.long,device=device)
    labels=torch.tensor([r['labels']+[IGNORE]*(width-len(r['labels'])) for r in encoded],
                        dtype=torch.long,device=device)
    mask=torch.tensor([[1]*len(r['input_ids'])+[0]*(width-len(r['input_ids'])) for r in encoded],
                      dtype=torch.long,device=device)
    return {'input_ids':ids,'labels':labels,'attention_mask':mask}


def _hf_loss(model,batched):
    """模型只提供logits，按本课掩码在外部shift一次；不调用内部labels loss。"""
    logits=model(input_ids=batched['input_ids'],attention_mask=batched['attention_mask'],
                 use_cache=False).logits
    return supervised_token_loss(logits.float(),batched['labels'])


@torch.no_grad()
def _hf_nll(model,encoded,tokenizer,device,batch_size=2):
    require(bool(encoded),'HF评估不能为空')
    pad_id=tokenizer.pad_token_id
    if pad_id is None:pad_id=tokenizer.eos_token_id
    require(type(pad_id) is int,'本地tokenizer必须提供pad或eos')
    old_mode=model.training;model.eval()
    total=0.0;count=0
    try:
        for start in range(0,len(encoded),batch_size):
            batch=_hf_token_batch(tokenizer,encoded[start:start+batch_size],pad_id,device)
            loss,parts=_hf_loss(model,batch)
            total+=parts['loss_sum'];count+=parts['target_tokens']
    finally:model.train(old_mode)
    return {'examples':len(encoded),'target_tokens':count,'teacher_forced_nll':total/count}


def hf_source_identity(audit:dict):
    require(audit.get('full_weight_content_hashed') is True,'必须完整哈希模型权重')
    return {'model_type':audit['model_type'],
            'weight_sha256':{item['file']:item['full_sha256'] for item in audit['weight_files']},
            'metadata_sha256':{item['file']:item['sha256'] for item in audit['small_files']}}


def _hf_checkpoint_files(path:Path):
    """只允许PEFT safetensors/配置/可选README；不接受pickle或代码工件。"""
    require(path.is_dir() and not path.is_symlink(),'HF adapter目录错误')
    names={p.name for p in path.iterdir()}
    require({'adapter_model.safetensors','adapter_config.json'}<=names,
            'PEFT未保存预期的safetensors/config文件')
    require(names<={'adapter_model.safetensors','adapter_config.json','README.md'},
            'adapter目录有非预期文件，拒绝发布')
    for p in path.iterdir():safe_file(p,128*2**20)
    cfg=read_json_bounded(path/'adapter_config.json',128*1024)
    require(type(cfg)is dict and cfg.get('peft_type')=='LORA','PEFT配置不是LoRA')
    return {p.name:compact_file_hash(p) for p in path.iterdir()}


def _hf_verify_adapter(base_folder:Path,adapter_dir:Path,tokenizer,encoded_dev,
                       device,expected_dev:float,target_modules:tuple[str,...],
                       *,fresh_checkpoint=True):
    from transformers import AutoModelForCausalLM
    from peft import PeftModel
    require(adapter_dir.is_dir(),'真实PEFT adapter必须是本地目录')
    local_model=AutoModelForCausalLM.from_pretrained(str(base_folder),local_files_only=True,
             trust_remote_code=False,use_safetensors=True,dtype=torch.float32)
    local_model=local_model.to(device)
    loaded=PeftModel.from_pretrained(local_model,str(adapter_dir),is_trainable=False,
                                     local_files_only=True)
    require(all(not p.requires_grad for p in loaded.parameters()),'重载PEFT不是推理冻结状态')
    measured=_hf_nll(loaded,encoded_dev,tokenizer,device)
    require(abs(measured['teacher_forced_nll']-expected_dev)<2e-3,
            'HF独立重载后的dev loss和发布承诺不符')
    return loaded,measured


def _hf_atomic_switch(root:Path,version:str,manifest_hash:str):
    require(SAFE_VERSION.fullmatch(version) is not None,'HF版本ID非法')
    folder=root/'versions'/version
    require(folder.is_dir() and not folder.is_symlink(),'HF目标版本未发布')
    require(compact_file_hash(folder/'manifest.json')==manifest_hash,'HF manifest摘要不一致')
    atomic_json(root/'CURRENT.json',{'format':VERSION,'backend':'peft',
                                      'version':version,'manifest_sha256':manifest_hash})


def _hf_manifest(root:Path,version:str,identity:dict,codec_identity:dict,expected_dev:float):
    folder=root/'versions'/version
    require(folder.is_dir() and not folder.is_symlink(),'HF版本目录非法')
    manifest=read_json_bounded(folder/'manifest.json',512*1024)
    require(manifest.get('version')==version and manifest.get('base_identity')==identity and
            manifest.get('tokenizer_contract')==codec_identity,'HF adapter绑定的基座或模板不符')
    require(type(manifest.get('dev_nll')) in (int,float) and
            abs(manifest['dev_nll']-expected_dev)<2e-3,'HF dev评估声明不符')
    recorded=manifest.get('file_hashes')
    require(recorded==_hf_checkpoint_files(folder/'adapter'),'HF PEFT文件摘要错误')
    return manifest


def run_local_peft_release(model_dir:str|Path,output_root:str|Path,
                            *,device_name='cpu',max_weight_bytes=256*2**20,
                            target_modules=('q_proj','v_proj'),rank=4,alpha=8.,
                            steps=4,learning_rate=1e-4,max_length=160,
                            min_dev_improvement=1e-4)->tuple[Path,dict]:
    """真实本地HF/PEFT路径：必须显式标志+可信文件+已安装兼容库。

    默认路径完全不会调用。无实际checkpoint/依赖时不会声称完成HF实测。
    """
    require(all(importlib.util.find_spec(x) is not None for x in
                ('transformers','peft','safetensors')),
            '缺少兼容的transformers/peft/safetensors；不会自动安装或下载')
    require(type(steps) is int and 1<=steps<=12,'真实PEFT演示最多12步')
    require(type(rank) is int and rank>0 and type(alpha) in (int,float) and math.isfinite(alpha)
            and alpha>0 and type(learning_rate) in (int,float) and
            math.isfinite(learning_rate) and learning_rate>0,'LoRA/HF学习率非法')
    require(type(max_length)is int and max_length>=32,'HF max_length非法')
    require(type(min_dev_improvement) in (int,float) and math.isfinite(min_dev_improvement)
            and min_dev_improvement>=0,'开发门槛非法')
    require(type(target_modules) in (tuple,list) and len(target_modules)>0 and
            len(set(target_modules))==len(target_modules) and
            all(type(s)is str and s and s.replace('_','').isalnum() for s in target_modules),
            '目标层必须为具体且唯一的nn.Linear后缀')
    source=ensure_local_directory(model_dir)
    validate_output_outside_model(source,output_root)
    before=inspect_local_training_candidate(source,max_weight_bytes=max_weight_bytes,
                                            full_weight_hash=True)
    identity=hf_source_identity(before)
    device=device_from_name(device_name)
    from transformers import AutoTokenizer,AutoModelForCausalLM
    from peft import LoraConfig,TaskType,get_peft_model
    tokenizer=AutoTokenizer.from_pretrained(str(source),local_files_only=True,
                                            trust_remote_code=False,use_fast=True)
    data=make_dataset(seed=7001)
    encoded={s:[_hf_encode_example(tokenizer,r,max_length) for r in rows]
             for s,rows in data.items()}
    codec_identity={'eos':tokenizer.eos_token_id,
                    'chat_template_sha256':hashlib.sha256(tokenizer.chat_template.encode()).hexdigest(),
                    'tokenizer_class':type(tokenizer).__name__,
                    'data_sha256':digest(data)}
    base=AutoModelForCausalLM.from_pretrained(str(source),local_files_only=True,
              trust_remote_code=False,use_safetensors=True,dtype=torch.float32)
    base=base.to(device).eval()
    matches=[(n,module) for n,module in base.named_modules()
             if any(n==s or n.endswith('.'+s) for s in target_modules)]
    require(bool(matches) and all(isinstance(module,nn.Linear) for _,module in matches),
            '指定目标层不是本课支持的nn.Linear')
    require(all(any(n==suffix or n.endswith('.'+suffix) for n,_ in matches)
                for suffix in target_modules),'至少一个目标后缀未匹配')
    lora_cfg=LoraConfig(r=rank,lora_alpha=alpha,lora_dropout=0.0,bias='none',
            target_modules=list(target_modules),task_type=TaskType.CAUSAL_LM)
    policy=get_peft_model(base,lora_cfg)
    trainable=[(n,p) for n,p in policy.named_parameters() if p.requires_grad]
    require(bool(trainable) and all(('lora_A.' in n or 'lora_B.' in n) for n,p in trainable),
            '存在非LoRA参数可训练')
    frozen={n:p for n,p in policy.named_parameters() if not p.requires_grad}
    frozen_before=state_hash(frozen)
    baseline=_hf_nll(policy,encoded['dev'],tokenizer,device)
    snapshot={n:p.detach().cpu().clone() for n,p in trainable}
    selected=0;best=baseline['teacher_forced_nll']
    optimizer=torch.optim.AdamW([p for _,p in trainable],lr=learning_rate,weight_decay=0.0)
    rng=random.Random(7003);history=[]
    pad_id=tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    for step in range(1,steps+1):
        batch_rows=rng.sample(encoded['train'],min(2,len(encoded['train'])))
        batch=_hf_token_batch(tokenizer,batch_rows,pad_id,device)
        policy.train();optimizer.zero_grad(set_to_none=True)
        loss,_=_hf_loss(policy,batch)
        require(bool(torch.isfinite(loss)),'HF训练loss非有限')
        loss.backward()
        require(all(p.grad is not None and bool(torch.isfinite(p.grad).all()) for _,p in trainable),
                'HF LoRA梯度缺失/非法')
        require(all(p.grad is None for p in frozen.values()),'冻结HF基座异常积累梯度')
        nn.utils.clip_grad_norm_([p for _,p in trainable],1.0,error_if_nonfinite=True)
        optimizer.step()
        measured=_hf_nll(policy,encoded['dev'],tokenizer,device)
        improved=measured['teacher_forced_nll']<best
        history.append({'step':step,'dev_nll':measured['teacher_forced_nll'],
                        'train_batch_nll':float(loss.detach()),'selected':improved})
        if improved:
            selected=step;best=measured['teacher_forced_nll']
            snapshot={n:p.detach().cpu().clone() for n,p in trainable}
    with torch.no_grad():
        for n,p in trainable:p.copy_(snapshot[n].to(p))
    require(state_hash(frozen)==frozen_before,'HF冻结基座发生变化')
    policy.eval()
    selected_dev=_hf_nll(policy,encoded['dev'],tokenizer,device)
    require(abs(selected_dev['teacher_forced_nll']-best)<2e-3,'恢复所选adapter后开发分数不匹配')
    require(hf_source_identity(inspect_local_training_candidate(source,
            max_weight_bytes=max_weight_bytes,full_weight_hash=True))==identity,
            '源checkpoint在训练时被修改')
    # 输出只写到模型目录外；真实adapter使用PEFT自身的safetensors格式而非toy的pt。
    output=new_run_dir(Path(output_root))
    (output/'versions').mkdir();(output/'.staging').mkdir()
    stage=output/'.staging'/'v0000';stage.mkdir()
    base_adapter=copy.deepcopy(snapshot)
    # 恢复零增量B，作为可回滚的基线adapter；仍要求独立重载评分一致。
    zero_parameters={name:p.detach().clone() for name,p in trainable}
    with torch.no_grad():
        for name,p in trainable:
            if 'lora_B.' in name:p.zero_()
    policy.save_pretrained(stage/'adapter',safe_serialization=True,save_embedding_layers=False)
    base_dev=_hf_nll(policy,encoded['dev'],tokenizer,device)
    require(abs(base_dev['teacher_forced_nll']-baseline['teacher_forced_nll'])<2e-3,
            'PEFT初始零增量B未恢复原始基座评分')
    with torch.no_grad():
        for name,p in trainable:p.copy_(zero_parameters[name])
    candidate_stage=output/'.staging'/'v0001';candidate_stage.mkdir()
    policy.save_pretrained(candidate_stage/'adapter',safe_serialization=True,save_embedding_layers=False)
    # 真实模型在独立对象上从原始本地safetensors重载，不能只检查参数形状。
    for version,stage_path,expected in [('v0000',stage,baseline['teacher_forced_nll']),
                                        ('v0001',candidate_stage,best)]:
        hashes=_hf_checkpoint_files(stage_path/'adapter')
        loaded,measure=_hf_verify_adapter(source,stage_path/'adapter',tokenizer,
                              encoded['dev'],device,expected,tuple(target_modules))
        del loaded
        write_json(stage_path/'manifest.json',{'format':VERSION,'backend':'peft','version':version,
                'base_identity':identity,'tokenizer_contract':codec_identity,
                'dev_nll':expected,'file_hashes':hashes,
                'selected_step':selected if version=='v0001' else 0,
                'target_modules':list(target_modules),
                'limits':'adapter-only; not merged weights; inference reload checked'})
        _fsync_directory(stage_path)
    os.replace(stage,output/'versions'/'v0000');_fsync_directory(output/'versions')
    manifest_hash=compact_file_hash(output/'versions'/'v0000'/'manifest.json')
    _hf_atomic_switch(output,'v0000',manifest_hash)
    gate=bool(selected>0 and best<=baseline['teacher_forced_nll']-min_dev_improvement)
    if gate:
        os.replace(candidate_stage,output/'versions'/'v0001');_fsync_directory(output/'versions')
        _hf_atomic_switch(output,'v0001',compact_file_hash(output/'versions'/'v0001'/'manifest.json'))
        # 回滚与恢复，不删除任一已发布版本。
        _hf_atomic_switch(output,'v0000',manifest_hash)
        _hf_atomic_switch(output,'v0001',compact_file_hash(output/'versions'/'v0001'/'manifest.json'))
    active='v0001' if gate else 'v0000'
    current=read_json_bounded(output/'CURRENT.json',16*1024)
    require(current['version']==active,'发布/回滚后的当前指针错误')
    _hf_manifest(output,active,identity,codec_identity,best if gate else baseline['teacher_forced_nll'])
    # 以上操作全部完成后才打开测试标签。
    test_before=_hf_nll(policy,encoded['test'],tokenizer,device) if gate else None
    fresh, _=_hf_verify_adapter(source,output/'versions'/active/'adapter',tokenizer,
        encoded['dev'],device,best if gate else baseline['teacher_forced_nll'],tuple(target_modules))
    test_final=_hf_nll(fresh,encoded['test'],tokenizer,device)
    del fresh
    source_after=hf_source_identity(inspect_local_training_candidate(source,
              max_weight_bytes=max_weight_bytes,full_weight_hash=True))
    require(identity==source_after,'真实本地模型目录被修改')
    report={'mode':'explicit_local_hf_peft','version':VERSION,
            'actual_local_model_training':True,'model_source_content_hashes_unchanged':True,
            'source_identity':identity,'tokenizer_contract':codec_identity,
            'target_modules':list(target_modules),
            'trainable_parameters':sum(p.numel() for _,p in trainable),
            'baseline_dev_nll':baseline['teacher_forced_nll'],'selected_dev_nll':best,
            'selected_step':selected,'steps':steps,'history':history,
            'gate_passed':gate,'current_version':active,
            'test_candidate_only_if_gate':test_before,
            'test_active_nll':test_final,
            'inference_reload_checked':True,
            'limitations':['trusted_local_files_not_a_security_sandbox',
                'no_downloads_no_remote_custom_code',
                'not_tested_with_actual_HF_model_in_build_environment',
                'no_quantization_no_distributed','no_exact_resume',
                'teacher_forced_loss_not_generation_accuracy',
                'not_CUDA_memory_benchmark','single_writer_same_filesystem_not_full_ACID']}
    write_json(output/'hf_release_report.json',report)
    print('显式本地PEFT输出:',output,'CURRENT:',active,'dev gate:',gate)
    return output,report

In [23]:
print('HF模式没有在本Notebook默认运行。')
print('需要先具备可信本地模型和对应依赖；无此条件不能声称真实PEFT微调已验证。')

HF模式没有在本Notebook默认运行。
需要先具备可信本地模型和对应依赖；无此条件不能声称真实PEFT微调已验证。


## 12. 命令行如何运行（不依赖Notebook）

默认CPU离线闭环：

```bash
python lesson_70_peft_adapter_release_and_rollback.py --checks-only
python lesson_70_peft_adapter_release_and_rollback.py
python lesson_70_peft_adapter_release_and_rollback.py --reload-run lesson_70_outputs/实际run目录
```

显式使用**你自己持有的可信本地小型HF Safetensors**（不会下载，不可用时应失败）：

```bash
python lesson_70_peft_adapter_release_and_rollback.py \
  --local-peft-model /absolute/path/to/trusted-local-model \
  --output-dir ./lesson_70_hf_outputs \
  --device cpu \
  --hf-steps 4 \
  --max-weight-bytes 268435456 \
  --target-modules q_proj,v_proj
```

模型路径不能是Hub ID或网络URL，输出必须在源模型目录外。运行前请确认依赖、支持的模块、硬件内存。**字节预检不是训练显存上界**，本课没有本地真实模型、量化或GPU速度实测。

In [24]:
def main(argv=None):
    parser=argparse.ArgumentParser(description=__doc__)
    group=parser.add_mutually_exclusive_group()
    group.add_argument('--checks-only',action='store_true')
    group.add_argument('--local-peft-model',metavar='EXISTING_LOCAL_HF_DIR')
    group.add_argument('--reload-run',metavar='TOY_RUN_DIR')
    parser.add_argument('--output-dir',default='lesson_70_outputs')
    parser.add_argument('--device',choices=['cpu','cuda','auto'],default='cpu')
    parser.add_argument('--steps',type=int,default=32)
    parser.add_argument('--batch-size',type=int,default=8)
    parser.add_argument('--min-dev-improvement',type=float,default=1e-4)
    parser.add_argument('--max-weight-bytes',type=int,default=256*2**20)
    parser.add_argument('--target-modules',default='q_proj,v_proj')
    parser.add_argument('--hf-steps',type=int,default=4)
    parser.add_argument('--hf-lr',type=float,default=1e-4)
    args=parser.parse_args(argv)
    torch.set_num_threads(1)
    if args.reload_run:
        print(json.dumps(reopen_toy_run(Path(args.reload_run),args.device),ensure_ascii=False,indent=2))
        return 0
    if args.local_peft_model:
        targets=tuple(x.strip() for x in args.target_modules.split(','))
        run_local_peft_release(args.local_peft_model,args.output_dir,device_name=args.device,
             max_weight_bytes=args.max_weight_bytes,target_modules=targets,
             steps=args.hf_steps,learning_rate=args.hf_lr,
             min_dev_improvement=args.min_dev_improvement)
        return 0
    checks=run_checks()
    print('固定检查通过：',checks['passed'],'项')
    if args.checks_only:return 0
    cfg=ReleaseConfig(steps=args.steps,batch_size=args.batch_size,
                      min_dev_improvement=args.min_dev_improvement)
    folder,_=run_toy_release(args.output_dir,args.device,cfg)
    write_json(folder/'checks.json',checks)
    return 0

## 13. 自动检查与失败注入

测试重点不是“模型loss能下降”这么简单，而是：

- 数据跨来源分组、Chat Template前缀、EOS、label mask与训练梯度；
- adapter-only 内容、基座哈希、键集合、NaN/错误形状；
- 发布前重新计算开发损失；不允许使用坏的基座、损坏文件或不够好的候选；
- 已发布版本不可覆盖、版本ID不能路径穿越；
- staged文件损坏、已发布manifest改变、CURRENT指向不存在版本时**明确拒绝**；
- 目录rename之前或指针更新之前中断，不能把旧CURRENT改成失败版本；
- 纯指针回滚/恢复，且独立重建模型的推理值一致。

它们只验证教学工程约定，不宣称系统可以抵御任意恶意本地文件、崩溃后的所有电源故障或并发操作。

In [25]:
class Checks:
    def __init__(self):self.names=[]
    def ok(self,name,condition):
        if not bool(condition):raise AssertionError(name)
        self.names.append(name)
    def close(self,name,a,b,atol=1e-8,rtol=1e-8):
        self.ok(name,torch.allclose(torch.as_tensor(a,dtype=torch.float64),
            torch.as_tensor(b,dtype=torch.float64),atol=atol,rtol=rtol))
    def raises(self,name,fn):
        try:fn()
        except (LessonError,ValueError,FileNotFoundError):self.names.append(name);return
        raise AssertionError(name+': 应拒绝却通过')


def _fixture(seed=7050):
    seed_all(seed)
    tok=ByteMergeChatTokenizer();data=make_dataset(seed)
    base=TinyChatLM(ToyConfig(tok.vocab_size,dim=16,heads=2,layers=1,ff_dim=32)).requires_grad_(False).eval()
    policy=inject_lora(base,rank=2,alpha=4.0)
    return tok,data,base,policy


def run_checks() -> dict:
    torch.set_num_threads(1);seed_all(7050)
    c=Checks()
    tok,data,base,policy=_fixture()
    c.ok('按来源划分32/8/8',[len(data[s]) for s in ('train','dev','test')]==[32,8,8])
    sources=[{r['source_id'] for r in data[s]} for s in ('train','dev','test')]
    c.ok('来源集合严格互斥',all(not sources[i]&sources[j] for i in range(3) for j in range(i+1,3)))
    c.ok('词表与控制token分离',tok.eos_token_id not in tok.encode('EOS'))
    c.ok('控制符字面输入不改变真实header',tok.apply_chat_template([{'role':'user','content':'<assistant>'}],tokenize=True,add_generation_prompt=True)[-2:]
         ==[tok.im_start_token_id,tok.role_ids['assistant']])
    row=data['train'][0]
    encoded=encode_training_row(tok,row)
    c.ok('提示与答案标签边界正确',encoded['labels'][:encoded['prompt_len']]==[IGNORE]*encoded['prompt_len'])
    c.ok('EOS计入监督',encoded['labels'][-1]==tok.eos_token_id)
    c.ok('正文与输入同位',encoded['labels'][encoded['prompt_len']:]==encoded['input_ids'][encoded['prompt_len']:])
    b=collate_training_rows([encode_training_row(tok,r) for r in data['train'][:4]],tok)
    c.ok('padding全忽略',bool((b['labels'][~b['attention_mask']]==IGNORE).all()))
    c.ok('prompt作为注意力上下文有效',bool(b['attention_mask'][:,0].all()))
    c.raises('数据来源跨划分拒绝',lambda:validate_dataset({**data,'dev':data['train'][:1]}))
    c.raises('超长样本不静默截断',lambda:encode_training_row(tok,row,12))
    baseline=evaluate_nll(policy,data['dev'],tok)
    c.ok('teacher-forced记录不冒充生成准确率',baseline['not_generation_accuracy'])
    c.ok('NLL有限非负',math.isfinite(baseline['teacher_forced_nll']) and baseline['teacher_forced_nll']>=0)
    before=state_hash(base.state_dict())
    c.ok('LoRA仅Q/V注入',sum(isinstance(m,LoRALinear) for m in policy.modules())==2)
    c.ok('只有A/B可训练',all(p.requires_grad==n.endswith(('.A','.B')) for n,p in policy.named_parameters()))
    c.ok('base与policy独立embedding存储',base.embed.weight.data_ptr()!=policy.embed.weight.data_ptr())
    with torch.no_grad():
        c.close('B零初始化与冻结基座输出一致',base(b['input_ids'],b['attention_mask']),
                policy(b['input_ids'],b['attention_mask']),atol=0,rtol=0)
    opt=torch.optim.AdamW(adapter_parameters(policy),lr=.01)
    first=toy_adapter_step(policy,opt,data['train'][:4],tok)
    c.ok('第一步A梯度为零',first['A_grad_sum']==0.)
    c.ok('第一步B梯度非零',first['B_grad_sum']>0.)
    second=toy_adapter_step(policy,opt,data['train'][:4],tok)
    c.ok('后续A得到梯度',second['A_grad_sum']>0.)
    c.ok('冻结基座内容不变',state_hash(base.state_dict())==before==base_hash(policy))
    c.ok('冻结基座无梯度',all(p.grad is None for p in base.parameters()))
    c.ok('保存只含A/B',all(k.endswith(('.A','.B')) for k in adapter_snapshot(policy)))
    c.ok('释放前adapter有更新',any(torch.count_nonzero(p).item()>0 for n,p in adapters(policy) if n.endswith('.B')))
    other=inject_lora(base,2,4.0)
    restore_adapter(other,adapter_snapshot(policy))
    c.ok('adapter状态可恢复',state_hash(adapter_snapshot(other))==state_hash(adapter_snapshot(policy)))
    c.raises('adapter键丢失拒绝',lambda:restore_adapter(other,dict(list(adapter_snapshot(policy).items())[1:])))
    bad=adapter_snapshot(policy);key=next(iter(bad));bad[key]=bad[key]+float('nan')
    c.raises('非有限adapter拒绝',lambda:restore_adapter(other,bad))
    c.raises('重复注入拒绝',lambda:inject_lora(policy,2,4.0))
    gen=greedy_toy(policy,tok,data['dev'][0],max_new_tokens=4)
    c.ok('生成含停止原因且不假装正确',gen['stop_reason'] in ('eos','max_tokens','context_limit'))
    c.ok('只有真实EOS与精确正文才计成功',gen['strict_match']==
         (gen['stop_reason']=='eos' and gen['valid_utf8'] and gen['text']==gen['expected']))
    c.raises('非法生成预算拒绝',lambda:greedy_toy(policy,tok,data['dev'][0],0))
    cfg=ReleaseConfig(seed=7050,steps=4,eval_every=2,batch_size=4)
    result=train_toy_candidate(policy,base,tok,data,cfg)
    c.ok('开发集选择含起点',result['selected_step'] in (0,2,4))
    c.ok('选定开发集NLL不高于初始',result['selected_dev_nll']<=result['baseline_dev_nll'])
    c.ok('选择不需要测试集结果','test' not in result)
    c.ok('训练后基座仍冻结',base_hash(policy)==before)
    independent=adapter_snapshot(policy);saved_identity=state_hash(independent)
    with torch.no_grad():
        dict(adapters(policy))[next(iter(independent))].add_(.03)
    c.ok('最佳adapter快照不是live张量视图',state_hash(independent)==saved_identity)
    restore_adapter(policy,independent)
    c.ok('恢复独立快照后张量身份一致',state_hash(adapter_snapshot(policy))==saved_identity)
    c.raises('配置不允许零步数',lambda:ReleaseConfig(steps=0).validate())
    c.raises('配置不允许负改善阈值',lambda:ReleaseConfig(min_dev_improvement=-1).validate())
    # 新版本仓库：先基线，之后候选；比较两次独立重载与回滚。
    with tempfile.TemporaryDirectory() as folder:
        root=Path(folder)/'release'
        store=ToyReleaseStore(root,base,tok,data,2,4.0,160)
        c.raises('尚未发布时CURRENT不可读取',lambda:store.current())
        base_policy=inject_lora(base,2,4.0)
        initial_nll=evaluate_nll(base_policy,data['dev'],tok)['teacher_forced_nll']
        stage=store.stage('v0000',base_policy,{'purpose':'baseline'})
        c.ok('保存只包含adapter与manifest',{p.name for p in stage.iterdir()}=={'adapter.pt','manifest.json'})
        blob=torch.load(stage/'adapter.pt',weights_only=True)
        c.ok('adapter权重本身不含base/head',set(blob['state'])==set(adapter_snapshot(base_policy)))
        c.ok('adapter文件小于基座',sum(p.numel()*p.element_size() for _,p in adapters(base_policy))<
             sum(p.numel()*p.element_size() for p in base.parameters()))
        _,manifest=store.validate('v0000')
        c.close('报告由独立重载再算开发loss',manifest['dev_nll'],initial_nll,atol=1e-6)
        published=store.promote('v0000',initial_nll,0.0)
        c.ok('基线正确发布',published['published'] and store.current()[0]=='v0000')
        c.ok('发布把staging移入versions',not (root/'.staging'/'v0000').exists())
        c.raises('不允许覆盖已有版本',lambda:store.stage('v0000',base_policy,{}))
        c.raises('拒绝版本路径穿越',lambda:store._version_folder('../v0000',True))
        c.raises('拒绝用test选版本',lambda:store.promote('v0000',0.,0.))
        # 使用训练候选测试开发门槛；阈值失败不应改变CURRENT。
        store.stage('v0001',policy,{'purpose':'candidate'})
        c.raises('过严开发门槛拒绝发布',lambda:store.promote('v0001',0.,0.))
        c.ok('门槛失败CURRENT不变',store.current()[0]=='v0000')
        after_policy,man=store.validate('v0001')
        candidate_dev=man['dev_nll']
        require(candidate_dev<initial_nll,'测试候选需要开发改善')
        stage_before=store.staging/'v0001'
        # 验证阶段报错时不会留下“发布成功”的当前指针。
        c.raises('故障在rename前阻止发布',lambda:store.promote('v0001',initial_nll,0.,fault='before_rename'))
        c.ok('rename前故障保留staging',stage_before.is_dir() and store.current()[0]=='v0000')
        promoted=store.promote('v0001',initial_nll,0.)
        c.ok('候选经独立验证后可发布',promoted['published'] and store.current()[0]=='v0001')
        c.ok('版本目录不含完整基座',{p.name for p in (root/'versions'/'v0001').iterdir()}=={'adapter.pt','manifest.json'})
        c.ok('发布加载的推理模型全冻结',all(not p.requires_grad for p in store.current()[1].parameters()))
        c.ok('当前manifest选择对象绑定正确',store.current()[2]['base_hash']==before)
        rolled=store.switch_to('v0000','rollback')
        c.ok('回滚只是更新CURRENT',rolled['from']=='v0001' and store.current()[0]=='v0000')
        c.ok('回滚不删除已验证候选',(root/'versions'/'v0001').is_dir())
        store2=ToyReleaseStore(root,base,tok,data,2,4.,160)
        c.ok('重新实例化仓库也能读取回滚指针',store2.current()[0]=='v0000')
        restored=store2.switch_to('v0001','restore')
        c.ok('恢复操作可再次切换',restored['to']=='v0001' and store2.current()[0]=='v0001')
        with torch.no_grad():
            probe=collate_training_rows([encode_training_row(tok,r) for r in data['dev'][:2]],tok)
            c.close('发布模型logits与原候选一致',store2.current()[1](probe['input_ids'],probe['attention_mask']),
                    policy(probe['input_ids'],probe['attention_mask']),atol=0,rtol=0)
        # 未提交候选被破坏不能影响指针。
        corrupt=store.stage('v0002',policy,{'purpose':'broken'})
        man_path=corrupt/'manifest.json';orig=read_json_bounded(man_path)
        broken=copy.deepcopy(orig);broken['base_hash']='bad'
        write_json(man_path,broken)
        c.raises('基座身份损坏被发现',lambda:store.validate('v0002'))
        c.ok('拒绝坏版本不修改CURRENT',store.current()[0]=='v0001')
        write_json(man_path,orig)
        c.ok('恢复manifest后可重读',store.validate('v0002')[1]['base_hash']==before)
        incorrect=copy.deepcopy(orig);incorrect['dev_nll']+=1.0
        write_json(man_path,incorrect)
        c.raises('宣称的dev NLL与新模型重算不符',lambda:store.validate('v0002'))
        write_json(man_path,orig)
        payload=corrupt/'adapter.pt';saved=payload.read_bytes()
        payload.write_bytes(saved+b'corrupted')
        c.raises('adapter文件字节改变被发现',lambda:store.validate('v0002'))
        payload.write_bytes(saved)
        c.ok('恢复原adapter后可重读',store.validate('v0002')[1]['file_sha256']==orig['file_sha256'])
        c.raises('故障在pointer前检测并保持旧CURRENT',lambda:store.promote('v0002',initial_nll,0.,fault='before_pointer'))
        c.ok('指针前故障留下未激活孤儿版本',store.current()[0]=='v0001' and
             (root/'versions'/'v0002').is_dir())
        c.raises('发布目录不可再次覆盖',lambda:store.stage('v0002',policy,{}))
        c.ok('CURRENT有可核对的manifest摘要',store.current()[2]['version']=='v0001')
        # 已发布版本的磁盘字节发生变化，必须拒绝回滚到受损版本。
        committed=root/'versions'/'v0002'/'manifest.json'
        current_bytes=committed.read_bytes();committed.write_bytes(current_bytes+b' ')
        c.raises('已发布版本被改动后切换拒绝',lambda:store.switch_to('v0002','rollback'))
        c.ok('切换失败原CURRENT仍可读取',store.current()[0]=='v0001')
        # CURRENT被损坏必须拒绝，而不是悄悄回退到最新的可见目录。
        pointer=root/'CURRENT.json';pointer_bytes=pointer.read_bytes()
        poison=read_json_bounded(pointer);poison['manifest_sha256']='0'*64
        write_json(pointer,poison)
        c.raises('CURRENT承诺的manifest摘要不符',lambda:store.current())
        pointer.write_bytes(pointer_bytes)
        poison=read_json_bounded(pointer);poison['version']='v9999'
        write_json(pointer,poison)
        c.raises('CURRENT指向不存在版本明确拒绝',lambda:store.current())
        pointer.write_bytes(pointer_bytes)
        c.ok('恢复CURRENT原字节后可读',store.current()[0]=='v0001')
        c.raises('rollback目标版本非法拒绝',lambda:store.switch_to('../v0000','rollback'))
        c.raises('rollback原因非法拒绝',lambda:store.switch_to('v0000','bad_reason'))
        audit=(root/'audit.jsonl').read_text().splitlines()
        c.ok('审计日志记录发布回滚恢复',
             [json.loads(x)['action'] for x in audit][:4]==['promote','promote','rollback','restore'])
        c.ok('没有覆盖/删除已验证v0000',(root/'versions'/'v0000').is_dir())
    # 明确受限的本地HF模式，不会在缺依赖时悄悄联网。
    c.raises('本地模型必须是现有本地目录',lambda:inspect_local_training_candidate('/nonexistent/lesson70_model'))
    if not all(importlib.util.find_spec(x) is not None for x in ('transformers','peft','safetensors')):
        c.raises('缺真实HF/PEFT依赖显式报错',lambda:run_local_peft_release('/nonexistent/lesson70_model','/tmp/local70'))
    return {'passed':len(c.names),'checks':c.names,
            'scope':'toy PyTorch CPU发布与回滚+数学/本地前置检查，不是实际HF checkpoint训练'}

In [26]:
check_report=run_checks()
print('固定检查通过:',check_report['passed'],'项。')
if RELEASE_DIR is not None:
    write_json(RELEASE_DIR/'checks.json',check_report)

固定检查通过: 83 项。


## 14. 练习区（检查默认关闭）

**练习A：开发集门槛。** 接收 `baseline_dev_nll`、`candidate_dev_nll` 和预先规定的最小改善值，返回布尔值。拒绝NaN、负值；不要读取test。

**练习B：独立LoRA快照。** 只保存A/B，每个tensor应 `.detach().cpu().clone()`，不能把活动参数的引用或共享存储直接放进“最佳模型”。

**练习C：指针切换小组件。** 在调用方完成版本身份与校验后，编写单次JSON指针原子替换，并拒绝非法vNNNN。这里只练习写指针，**完整生产流程仍需要ToyReleaseStore那样先验证目标版本**。

In [27]:
def check_exercise_a(fn):
    tok,data,base,policy=_fixture(7111)
    encoded=encode_training_row(tok,data['dev'][0])
    baseline=eval_nll=4.0
    got=fn(baseline,3.7,0.01)
    require(got is True,'改善超过阈值应准许')
    require(fn(4.0,4.0,0.0) is True,'门槛零且持平允许')
    require(fn(4.0,4.0,0.01) is False,'没有足够改善应该拒绝')
    require(fn(4.0,4.1,0.0) is False,'开发指标退步应该拒绝')
    try:fn(4.0,float('nan'),0.0)
    except ValueError:pass
    else:raise LessonError('NaN应拒绝')
    print('练习A：dev发布门槛核对成功。')


def check_exercise_b(fn):
    tok,data,base,policy=_fixture(7112)
    snap=fn(policy)
    require(type(snap)is dict and set(snap)==set(adapter_snapshot(policy)),
            '快照名称不匹配')
    require(all(torch.equal(snap[k],adapter_snapshot(policy)[k]) for k in snap),
            '快照初始值不一致')
    with torch.no_grad():
        for _,p in adapters(policy):p.add_(1)
    require(all(torch.equal(snap[k],v) for k,v in snap.items()),'检查输入读取失败')
    require(any(not torch.equal(snap[k],dict(adapters(policy))[k].detach().cpu()) for k in snap),
            '快照共享了live参数的存储')
    print('练习B：LoRA张量快照独立性核对成功。')


def check_exercise_c(fn):
    with tempfile.TemporaryDirectory() as tmp:
        ptr=Path(tmp)/'CURRENT.json'
        fn(ptr,'v0000','a'*64)
        old=read_json_bounded(ptr)
        require(old['version']=='v0000' and old['manifest_sha256']=='a'*64,'首次发布错误')
        fn(ptr,'v0001','b'*64)
        new=read_json_bounded(ptr)
        require(new['version']=='v0001' and new['manifest_sha256']=='b'*64,'第二次切换错误')
        fn(ptr,'v0000','a'*64)
        require(read_json_bounded(ptr)==old,'回滚没有恢复指针')
        try:fn(ptr,'../etc','a'*64)
        except ValueError:pass
        else:raise LessonError('非法version必须拒绝')
        require(read_json_bounded(ptr)==old,'失败操作修改了CURRENT')
        require(not list(ptr.parent.glob('CURRENT.json.tmp*')),'遗留未提交临时文件')
    print('练习C：原子指针更新与回滚核对成功。')

In [28]:
def dev_release_gate_ex(baseline_dev_nll,candidate_dev_nll,minimum_improvement):
    raise NotImplementedError('练习A：请实现开发集门槛')

RUN_EXERCISE_A=False
if RUN_EXERCISE_A:check_exercise_a(dev_release_gate_ex)

In [29]:
def snapshot_lora_ex(policy):
    raise NotImplementedError('练习B：请实现独立LoRA快照')

RUN_EXERCISE_B=False
if RUN_EXERCISE_B:check_exercise_b(snapshot_lora_ex)

In [30]:
def write_current_ex(pointer,version,manifest_sha256):
    raise NotImplementedError('练习C：请实现单写者条件下的CURRENT原子替换')

RUN_EXERCISE_C=False
if RUN_EXERCISE_C:check_exercise_c(write_current_ex)

## 15. 独立参考答案

先完成自己的练习并开启检查，再核对独立参考函数。这些答案不会覆盖你的空白函数。

In [31]:
def answer_a(baseline_dev_nll,candidate_dev_nll,minimum_improvement):
    values=[baseline_dev_nll,candidate_dev_nll,minimum_improvement]
    require(all(type(v) in (int,float) and math.isfinite(v) and v>=0 for v in values),
            '门槛输入必须有限非负')
    return candidate_dev_nll<=baseline_dev_nll-minimum_improvement


def answer_b(policy):
    return {name:param.detach().cpu().clone() for name,param in adapters(policy)}


def answer_c(pointer,version,manifest_sha256):
    require(type(version)is str and SAFE_VERSION.fullmatch(version),'版本ID非法')
    require(type(manifest_sha256)is str and re.fullmatch(r'[0-9a-f]{64}',manifest_sha256),
            'manifest摘要格式非法')
    atomic_json(Path(pointer),{'version':version,'manifest_sha256':manifest_sha256})

In [32]:
check_exercise_a(answer_a)
check_exercise_b(answer_b)
check_exercise_c(answer_c)

练习A：dev发布门槛核对成功。
练习B：LoRA张量快照独立性核对成功。
练习C：原子指针更新与回滚核对成功。


## 16. 本课验收与下一课

应能够解释：

1. 为什么发布必须是**先保存stage → 验证 → 登记 → 更新CURRENT**，而不直接覆盖唯一adapter文件？
2. 为什么把 `B` 矩阵初始化为零时，初始策略输出等于基座？
3. 开发集门槛通过，为什么还必须记录独立测试的真实生成？
4. 为什么“base哈希相同＋LoRA名称、shape一致＋重新计算输出一致”比“rank相同”更可信？
5. `os.replace` 能保证什么，不能保证什么？为什么不承诺跨文件系统/并发/断电完全一致？
6. 为什么Toy模型另存 `base_for_replay.pt`，但真正PEFT adapter目录只含 `safetensors`、配置与可选README？
7. 为什么对用户没有提供的本地HF模型，不能声称已经完成PEFT真机训练？

**第71课预告：LoRA权重合并、反合并与部署等价性验证。** 将比较运行时adapter与merge后权重的数值误差、可逆性边界，以及推理发布前的验收。

### 官方参考资料

- [PEFT checkpoint格式：基座仍必需](https://huggingface.co/docs/peft/main/developer_guides/checkpoint)
- [PEFT PeftModel.from_pretrained、save_pretrained](https://huggingface.co/docs/peft/main/package_reference/peft_model)
- [LoRA target_modules、rank、alpha](https://huggingface.co/docs/peft/main/package_reference/lora)
- [Transformers local_files_only、trust_remote_code](https://huggingface.co/docs/transformers/model_doc/auto)
- [Chat Template：训练/推理对齐](https://huggingface.co/docs/transformers/main/chat_templating)
- [PyTorch持久化与可信加载边界](https://docs.pytorch.org/docs/stable/notes/serialization.html)

以上是实现接口与正确性依据，不是本课已经在RTX 3060或真实预训练权重上测得成绩的证据。